In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_26.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_33.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_49.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_44.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_38.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_22.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_30.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_57.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_46.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_58.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_21.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_59.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_32.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_69.csv
/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/data_19.csv
/kaggle/input/datasets/vigneshvenkateswa

In [2]:
"""
================================================================================
BOT-IoT LARGE-SCALE INTRUSION DETECTION SYSTEM
================================================================================
Production-Grade Kaggle Notebook for IoT Cybersecurity AI Research

Author: Senior ML Engineering Pipeline
Dataset: Bot-IoT (~72M rows, 74 CSV files)
Framework: LightGBM + Optional Deep Autoencoder + SHAP Explainability

Engineering Priorities:
1. Stability & Memory Efficiency
2. Scalability via Chunked Processing
3. Generalization & Cross-Dataset Hooks
4. Explainability & Research Rigor
5. Robustness & Recovery Mechanisms

================================================================================
"""

# ==============================================================================
# SECTION 0 — NOTEBOOK CONFIGURATION & GLOBALS
# ==============================================================================

# Toggle switches for production flexibility
USE_GPU = True                    # LightGBM GPU mode (auto-detected)
USE_AUTOENCODER = False            # Deep anomaly detector (GPU-dependent)
FULL_DATA = False                  # Process full dataset vs. stratified subset
USE_MLFLOW = False                 # Experiment tracking (lightweight fallback)
SAMPLE_FRACTION = 0.05             # Fraction for subset mode (5% = ~3.6M rows)
RANDOM_SEED = 42
N_OPTUNA_TRIALS = 10               # Hyperparameter optimization trials
nrows_per_chunk = 500_000          # Chunk size for CSV reading

# Paths
DATA_PATH = '/kaggle/input/datasets/vigneshvenkateswaran/bot-iot/'
WORKING_PATH = '/kaggle/working/'
PARQUET_PATH = WORKING_PATH + 'parquet_data/'
MODEL_PATH = WORKING_PATH + 'models/'
CHECKPOINT_PATH = WORKING_PATH + 'checkpoints/'
LOG_PATH = WORKING_PATH + 'logs/'

# Autoencoder config (dynamic based on VRAM)
AUTOENCODER_CONFIG = {
    'hidden_dims': [128, 64, 32, 64, 128],
    'dropout': 0.2,
    'learning_rate': 1e-3,
    'max_epochs': 50,
    'patience': 10,
    'batch_size': 512,  # dynamically adjusted
}

In [3]:
# ==============================================================================
# SECTION 1 — ENVIRONMENT SETUP
# ==============================================================================

import subprocess
import sys
import os
import gc
import time
import json
import pickle
import warnings
import logging
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from typing import Dict, List, Tuple, Optional, Union, Callable, Any
import hashlib
import copy

import numpy as np
import pandas as pd

# Memory and system
import psutil

# Visualization
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for Kaggle
plt.style.use('seaborn-v0_8-whitegrid')
import seaborn as sns

# Plotly for interactive research visualizations
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff

# ML Core
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, RobustScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, 
    confusion_matrix, classification_report,
    precision_recall_curve, roc_curve
)
from sklearn.utils.class_weight import compute_class_weight

# LightGBM
import lightgbm as lgb

# Optuna for HPO
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# SHAP for explainability
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("[WARN] SHAP not available, skipping explainability")

# Deep Learning (conditional)
TORCH_AVAILABLE = False
if USE_AUTOENCODER:
    try:
        import torch
        import torch.nn as nn
        import torch.optim as optim
        from torch.utils.data import Dataset, DataLoader
        TORCH_AVAILABLE = True
    except ImportError:
        print("[WARN] PyTorch not available, disabling autoencoder")

# Polars for efficient processing (preferred over pandas)
try:
    import polars as pl
    POLARS_AVAILABLE = True
    print("[INFO] Using Polars for high-performance processing")
except ImportError:
    POLARS_AVAILABLE = False
    print("[WARN] Polars not available, falling back to optimized pandas")

# PyArrow for Parquet I/O
try:
    import pyarrow as pa
    import pyarrow.parquet as pq
    import pyarrow.compute as pc
    PYARROW_AVAILABLE = True
except ImportError:
    PYARROW_AVAILABLE = False

# MLflow or lightweight logging
try:
    import mlflow
    import mlflow.lightgbm
    MLFLOW_AVAILABLE = True
except ImportError:
    MLFLOW_AVAILABLE = False

# Suppress warnings for clean output
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# ==============================================================================
# REPRODUCIBILITY SETUP
# ==============================================================================

def set_seed(seed: int = 42) -> None:
    """Set all random seeds for reproducibility."""
    np.random.seed(seed)
    if TORCH_AVAILABLE:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    # Python hash seed (requires env var before process start)
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"[SEED] Random seed set to {seed}")

set_seed(RANDOM_SEED)

# ==============================================================================
# LOGGING INFRASTRUCTURE
# ==============================================================================

class ExperimentLogger:
    """
    Lightweight experiment tracking with MLflow fallback.
    Tracks: parameters, metrics, runtime, dataset sizes, feature counts.
    """

    def __init__(self, experiment_name: str = "bot_iot_ids", use_mlflow: bool = False):
        self.experiment_name = experiment_name
        self.use_mlflow = use_mlflow and MLFLOW_AVAILABLE
        self.run_id = None
        self.metrics = defaultdict(list)
        self.params = {}
        self.start_time = time.time()
        self.events = []

        # Setup file logging
        os.makedirs(LOG_PATH, exist_ok=True)
        log_file = os.path.join(LOG_PATH, f"{experiment_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log")

        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s [%(levelname)s] %(message)s',
            handlers=[
                logging.FileHandler(log_file),
                logging.StreamHandler(sys.stdout)
            ]
        )
        self.logger = logging.getLogger(experiment_name)

        if self.use_mlflow:
            mlflow.set_experiment(experiment_name)
            mlflow.start_run()
            self.run_id = mlflow.active_run().info.run_id
            self.logger.info(f"MLflow tracking started: run_id={self.run_id}")
        else:
            self.logger.info("Using lightweight structured logging (MLflow unavailable)")

    def log_param(self, key: str, value: Any) -> None:
        """Log a parameter."""
        self.params[key] = value
        if self.use_mlflow:
            mlflow.log_param(key, value)
        self.logger.info(f"[PARAM] {key} = {value}")

    def log_metric(self, key: str, value: float, step: Optional[int] = None) -> None:
        """Log a metric (optionally with step for time series)."""
        self.metrics[key].append((step, value) if step is not None else value)
        if self.use_mlflow:
            mlflow.log_metric(key, value, step=step)
        self.logger.info(f"[METRIC] {key} = {value:.6f}" + (f" (step={step})" if step else ""))

    def log_event(self, event: str, details: Dict = None) -> None:
        """Log a structured event with timestamp."""
        entry = {
            'timestamp': datetime.now().isoformat(),
            'event': event,
            'details': details or {},
            'elapsed_sec': time.time() - self.start_time
        }
        self.events.append(entry)
        self.logger.info(f"[EVENT] {event} | {json.dumps(details) if details else ''}")

    def log_artifact(self, local_path: str) -> None:
        """Log a file artifact."""
        if self.use_mlflow:
            mlflow.log_artifact(local_path)
        self.logger.info(f"[ARTIFACT] Saved: {local_path}")

    def get_summary(self) -> Dict:
        """Get experiment summary."""
        return {
            'experiment': self.experiment_name,
            'run_id': self.run_id,
            'duration_sec': time.time() - self.start_time,
            'params': self.params,
            'metrics': {k: v[-1] if isinstance(v, list) and v else v for k, v in self.metrics.items()},
            'event_count': len(self.events)
        }

    def end(self) -> None:
        """End the experiment run."""
        if self.use_mlflow:
            mlflow.end_run()
        self.logger.info(f"[DONE] Experiment completed. Duration: {time.time() - self.start_time:.1f}s")
        # Save structured log
        summary_path = os.path.join(LOG_PATH, f"summary_{self.experiment_name}.json")
        with open(summary_path, 'w') as f:
            json.dump(self.get_summary(), f, indent=2, default=str)

# Initialize global logger
logger = ExperimentLogger("bot_iot_ids_v1", use_mlflow=USE_MLFLOW)

# ==============================================================================
# MEMORY MONITORING UTILITIES
# ==============================================================================

class MemoryMonitor:
    """
    Continuous memory monitoring for large-scale processing.
    Tracks RAM, dataframe sizes, parquet storage, processing throughput.
    """

    def __init__(self, log_interval: int = 30):
        self.log_interval = log_interval
        self.last_log_time = time.time()
        self.peak_ram_gb = 0.0
        self.dataframe_sizes = []
        self.throughput_records = []
        self.process = psutil.Process()

    def get_ram_usage(self) -> Dict[str, float]:
        """Get current RAM usage statistics."""
        mem = psutil.virtual_memory()
        proc_mem = self.process.memory_info()
        return {
            'total_gb': mem.total / (1024**3),
            'available_gb': mem.available / (1024**3),
            'used_gb': mem.used / (1024**3),
            'process_rss_gb': proc_mem.rss / (1024**3),
            'process_vms_gb': proc_mem.vms / (1024**3),
            'percent': mem.percent
        }

    def get_dataframe_size(self, df, name: str = "dataframe") -> Dict:
        """Estimate memory usage of a dataframe."""
        if POLARS_AVAILABLE and isinstance(df, pl.DataFrame):
            mem_bytes = df.estimated_size()
        elif isinstance(df, pd.DataFrame):
            mem_bytes = df.memory_usage(deep=True).sum()
        else:
            mem_bytes = 0

        size_mb = mem_bytes / (1024**2)
        self.dataframe_sizes.append({'name': name, 'size_mb': size_mb, 'timestamp': time.time()})
        return {'name': name, 'size_mb': size_mb, 'size_gb': size_mb / 1024}

    def log(self, force: bool = False) -> Dict:
        """Log current memory state if interval elapsed or forced."""
        current_time = time.time()
        if force or (current_time - self.last_log_time >= self.log_interval):
            ram = self.get_ram_usage()
            self.peak_ram_gb = max(self.peak_ram_gb, ram['process_rss_gb'])

            logger.log_event("memory_status", {
                'ram_used_gb': round(ram['used_gb'], 2),
                'ram_available_gb': round(ram['available_gb'], 2),
                'process_rss_gb': round(ram['process_rss_gb'], 2),
                'peak_ram_gb': round(self.peak_ram_gb, 2),
                'ram_percent': ram['percent']
            })
            self.last_log_time = current_time
            return ram
        return None

    def record_throughput(self, rows_processed: int, elapsed_sec: float, operation: str) -> None:
        """Record processing throughput."""
        throughput = rows_processed / elapsed_sec if elapsed_sec > 0 else 0
        self.throughput_records.append({
            'operation': operation,
            'rows': rows_processed,
            'elapsed_sec': elapsed_sec,
            'rows_per_sec': throughput,
            'timestamp': time.time()
        })
        logger.log_event("throughput", {
            'operation': operation,
            'rows': rows_processed,
            'rows_per_sec': round(throughput, 1),
            'elapsed_sec': round(elapsed_sec, 2)
        })

    def print_summary(self) -> None:
        """Print memory and throughput summary."""
        print("\n" + "="*60)
        print("MEMORY & THROUGHPUT SUMMARY")
        print("="*60)
        print(f"Peak Process RAM: {self.peak_ram_gb:.2f} GB")
        if self.throughput_records:
            avg_throughput = np.mean([r['rows_per_sec'] for r in self.throughput_records])
            print(f"Average Throughput: {avg_throughput:,.0f} rows/sec")
        if self.dataframe_sizes:
            total_df_mb = sum(d['size_mb'] for d in self.dataframe_sizes)
            print(f"Total DataFrame Memory Tracked: {total_df_mb:.1f} MB")
        print("="*60)

# Initialize global memory monitor
mem_monitor = MemoryMonitor(log_interval=60)


# ==============================================================================
# SYSTEM INFO PRINT
# ==============================================================================

def detect_gpu() -> Tuple[bool, str]:
    """
    Detect GPU availability and return status + info string.
    Separated from print_system_info() for clarity and testability.
    """
    if not TORCH_AVAILABLE:
        return False, "PyTorch not installed"
    
    if not torch.cuda.is_available():
        return False, "No CUDA device detected"
    
    # GPU is available
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    gpu_info = f"{gpu_name} ({gpu_mem_gb:.1f} GB)"
    
    return True, gpu_info


def print_system_info() -> None:
    """Print comprehensive system information."""
    print("\n" + "="*60)
    print("SYSTEM INFORMATION")
    print("="*60)

    # CPU
    cpu_count = psutil.cpu_count(logical=True)
    cpu_freq = psutil.cpu_freq()
    print(f"CPU: {cpu_count} logical cores")
    if cpu_freq:
        print(f"CPU Frequency: {cpu_freq.max:.0f} MHz (max)")

    # RAM
    mem = psutil.virtual_memory()
    print(f"Total RAM: {mem.total / (1024**3):.1f} GB")
    print(f"Available RAM: {mem.available / (1024**3):.1f} GB")

    # GPU — use dedicated detector
    gpu_available, gpu_info = detect_gpu()
    print(f"GPU: {gpu_info}")
    
    # Update global flag cleanly
    global USE_GPU
    USE_GPU = gpu_available
    print(f"[INFO] USE_GPU set to: {USE_GPU}")

    # Disk
    disk = psutil.disk_usage('/')
    print(f"Disk Total: {disk.total / (1024**3):.1f} GB")
    print(f"Disk Free: {disk.free / (1024**3):.1f} GB")

    # Libraries
    print(f"\nLibrary Versions:")
    print(f"  NumPy: {np.__version__}")
    print(f"  Pandas: {pd.__version__}")
    print(f"  LightGBM: {lgb.__version__}")
    import sklearn
    print(f"  Scikit-learn: {sklearn.__version__}")
    print(f"  Polars: {pl.__version__ if POLARS_AVAILABLE else 'N/A'}")
    print(f"  PyArrow: {pa.__version__ if PYARROW_AVAILABLE else 'N/A'}")
    print(f"  Optuna: {optuna.__version__}")
    print(f"  SHAP: {shap.__version__ if SHAP_AVAILABLE else 'N/A'}")
    print(f"  Torch: {torch.__version__ if TORCH_AVAILABLE else 'N/A'}")
    print("="*60)

    logger.log_param("cpu_cores", cpu_count)
    logger.log_param("total_ram_gb", round(mem.total / (1024**3), 1))
    logger.log_param("gpu_available", USE_GPU)
    logger.log_param("gpu_name", gpu_info.split(' (')[0] if USE_GPU else "None")
    logger.log_param("polars_available", POLARS_AVAILABLE)

print_system_info()
mem_monitor.log(force=True)


# ==============================================================================
# CHECKPOINT & RECOVERY SYSTEM
# ==============================================================================

class CheckpointManager:
    """
    Robust checkpointing for resumable preprocessing.
    Handles partial parquet recovery and intermediate caching.
    """

    def __init__(self, checkpoint_dir: str = CHECKPOINT_PATH):
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(checkpoint_dir, exist_ok=True)
        self.state_file = os.path.join(checkpoint_dir, "pipeline_state.json")
        self.state = self._load_state()

    def _load_state(self) -> Dict:
        """Load previous checkpoint state."""
        if os.path.exists(self.state_file):
            with open(self.state_file, 'r') as f:
                return json.load(f)
        return {
            'csv_files_processed': [],
            'parquet_files_created': [],
            'last_successful_step': None,
            'feature_engineering_done': False,
            'train_test_split_done': False,
            'model_trained': False,
            'timestamp': None
        }

    def save_state(self) -> None:
        """Save current state to disk."""
        self.state['timestamp'] = datetime.now().isoformat()
        with open(self.state_file, 'w') as f:
            json.dump(self.state, f, indent=2)
        logger.log_event("checkpoint_saved", {'state': self.state})

    def mark_csv_processed(self, csv_file: str, parquet_file: str) -> None:
        """Mark a CSV file as successfully converted."""
        if csv_file not in self.state['csv_files_processed']:
            self.state['csv_files_processed'].append(csv_file)
        if parquet_file not in self.state['parquet_files_created']:
            self.state['parquet_files_created'].append(parquet_file)
        self.save_state()

    def is_csv_processed(self, csv_file: str) -> bool:
        """Check if CSV was already processed."""
        return csv_file in self.state['csv_files_processed']

    def mark_step_complete(self, step: str) -> None:
        """Mark a pipeline step as complete."""
        self.state[f'{step}_done'] = True
        self.state['last_successful_step'] = step
        self.save_state()

    def is_step_complete(self, step: str) -> bool:
        """Check if a step is complete."""
        return self.state.get(f'{step}_done', False)

    def get_processed_parquets(self) -> List[str]:
        """Get list of successfully created parquet files."""
        return [p for p in self.state['parquet_files_created'] if os.path.exists(p)]

    def clear(self) -> None:
        """Clear all checkpoints (fresh start)."""
        self.state = {
            'csv_files_processed': [],
            'parquet_files_created': [],
            'last_successful_step': None,
            'feature_engineering_done': False,
            'train_test_split_done': False,
            'model_trained': False,
            'timestamp': None
        }
        self.save_state()

# Initialize checkpoint manager
ckpt = CheckpointManager()

[INFO] Using Polars for high-performance processing
[SEED] Random seed set to 42
2026-05-11 20:05:27,510 [INFO] Using lightweight structured logging (MLflow unavailable)

SYSTEM INFORMATION
CPU: 4 logical cores
CPU Frequency: 0 MHz (max)
Total RAM: 31.4 GB
Available RAM: 30.2 GB
GPU: PyTorch not installed
[INFO] USE_GPU set to: False
Disk Total: 8062.4 GB
Disk Free: 1220.5 GB

Library Versions:
  NumPy: 2.0.2
  Pandas: 2.3.3
  LightGBM: 4.6.0
  Scikit-learn: 1.6.1
  Polars: 1.35.2
  PyArrow: 23.0.1
  Optuna: 4.8.0
  SHAP: 0.50.0
  Torch: N/A
2026-05-11 20:05:27,514 [INFO] [PARAM] cpu_cores = 4
2026-05-11 20:05:27,515 [INFO] [PARAM] total_ram_gb = 31.4
2026-05-11 20:05:27,515 [INFO] [PARAM] gpu_available = False
2026-05-11 20:05:27,516 [INFO] [PARAM] gpu_name = None
2026-05-11 20:05:27,517 [INFO] [PARAM] polars_available = True
2026-05-11 20:05:27,518 [INFO] [EVENT] memory_status | {"ram_used_gb": 0.71, "ram_available_gb": 30.2, "process_rss_gb": 0.45, "peak_ram_gb": 0.45, "ram_percent"

In [4]:
# ==============================================================================
# SECTION 2 — DATASET DISCOVERY
# ==============================================================================

def discover_dataset(data_path: str = DATA_PATH) -> Dict:
    """
    Automatically discover all CSV files in the dataset directory.
    Returns metadata about discovered files.
    """
    print("\n" + "="*60)
    print("DATASET DISCOVERY")
    print("="*60)

    if not os.path.exists(data_path):
        raise FileNotFoundError(f"Dataset path not found: {data_path}")

    # Find all CSV files
    all_files = sorted([
        f for f in os.listdir(data_path) 
        if f.endswith('.csv')
    ])

    # Exclude data_names.csv from training
    csv_files = [f for f in all_files if f != 'data_names.csv']
    data_names_file = [f for f in all_files if f == 'data_names.csv']

    # Sort naturally (data_1.csv, data_2.csv, ..., data_10.csv, ...)
    def natural_sort_key(filename):
        import re
        return [int(text) if text.isdigit() else text.lower() 
                for text in re.split('([0-9]+)', filename)]

    csv_files.sort(key=natural_sort_key)

    # Calculate total size
    total_size_bytes = sum(
        os.path.getsize(os.path.join(data_path, f)) 
        for f in csv_files
    )
    total_size_gb = total_size_bytes / (1024**3)

    # Sample filenames
    sample_files = csv_files[:5] + (['...'] if len(csv_files) > 5 else []) + csv_files[-3:]

    print(f"Total CSV files found: {len(csv_files)}")
    print(f"Excluded: data_names.csv")
    print(f"Estimated total size: {total_size_gb:.2f} GB")
    print(f"Sample files: {sample_files}")
    print("="*60)

    logger.log_param("total_csv_files", len(csv_files))
    logger.log_param("dataset_size_gb", round(total_size_gb, 2))
    logger.log_param("sample_fraction", SAMPLE_FRACTION)
    logger.log_param("full_data_mode", FULL_DATA)

    return {
        'csv_files': csv_files,
        'data_names': data_names_file,
        'total_files': len(csv_files),
        'total_size_gb': total_size_gb,
        'data_path': data_path
    }

# Execute discovery
dataset_info = discover_dataset()
mem_monitor.log(force=True)


DATASET DISCOVERY
Total CSV files found: 74
Excluded: data_names.csv
Estimated total size: 13.97 GB
Sample files: ['data_1.csv', 'data_2.csv', 'data_3.csv', 'data_4.csv', 'data_5.csv', '...', 'data_72.csv', 'data_73.csv', 'data_74.csv']
2026-05-11 20:05:27,609 [INFO] [PARAM] total_csv_files = 74
2026-05-11 20:05:27,610 [INFO] [PARAM] dataset_size_gb = 13.97
2026-05-11 20:05:27,610 [INFO] [PARAM] sample_fraction = 0.05
2026-05-11 20:05:27,611 [INFO] [PARAM] full_data_mode = False
2026-05-11 20:05:27,612 [INFO] [EVENT] memory_status | {"ram_used_gb": 0.71, "ram_available_gb": 30.2, "process_rss_gb": 0.45, "peak_ram_gb": 0.45, "ram_percent": 3.7}


{'total_gb': 31.350616455078125,
 'available_gb': 30.19571304321289,
 'used_gb': 0.7092628479003906,
 'process_rss_gb': 0.4537391662597656,
 'process_vms_gb': 3.547863006591797,
 'percent': 3.7}

In [5]:
# ==============================================================================
# SECTION 3 — EFFICIENT CSV -> PARQUET PIPELINE
# ==============================================================================

def optimize_dtypes_polars(df: pl.DataFrame) -> pl.DataFrame:
    """
    Aggressively optimize Polars dataframe dtypes for memory efficiency.
    Downcast numeric columns, encode categoricals.
    """
    optimized = []

    for col_name in df.columns:
        series = df[col_name]
        dtype = series.dtype

        # Skip if already optimal
        if dtype in [pl.Categorical, pl.Boolean]:
            optimized.append(series)
            continue

        # Integer downcasting
        if dtype in [pl.Int64, pl.Int32, pl.Int16]:
            min_val = series.min()
            max_val = series.max()

            if min_val is not None and max_val is not None:
                if min_val >= 0:
                    # Unsigned
                    if max_val <= 255:
                        optimized.append(series.cast(pl.UInt8))
                    elif max_val <= 65535:
                        optimized.append(series.cast(pl.UInt16))
                    elif max_val <= 4294967295:
                        optimized.append(series.cast(pl.UInt32))
                    else:
                        optimized.append(series)
                else:
                    # Signed
                    if min_val >= -128 and max_val <= 127:
                        optimized.append(series.cast(pl.Int8))
                    elif min_val >= -32768 and max_val <= 32767:
                        optimized.append(series.cast(pl.Int16))
                    elif min_val >= -2147483648 and max_val <= 2147483647:
                        optimized.append(series.cast(pl.Int32))
                    else:
                        optimized.append(series)
            else:
                optimized.append(series)

        # Float downcasting
        elif dtype == pl.Float64:
            optimized.append(series.cast(pl.Float32))

        # String to categorical for low cardinality
        elif dtype == pl.Utf8:
            n_unique = series.n_unique()
            n_total = len(series)
            if n_unique / n_total < 0.5 and n_unique < 10000:
                optimized.append(series.cast(pl.Categorical))
            else:
                optimized.append(series)

        else:
            optimized.append(series)

    return pl.DataFrame(optimized)


def optimize_dtypes_pandas(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggressively optimize pandas dataframe dtypes for memory efficiency.
    Fallback when Polars is unavailable.
    """
    optimized = df.copy()

    for col in optimized.columns:
        col_type = optimized[col].dtype

        if col_type == 'object':
            num_unique = optimized[col].nunique()
            num_total = len(optimized[col])
            if num_unique / num_total < 0.5:
                optimized[col] = optimized[col].astype('category')

        elif col_type == 'int64':
            c_min = optimized[col].min()
            c_max = optimized[col].max()
            if c_min >= 0:
                if c_max < 255:
                    optimized[col] = optimized[col].astype(np.uint8)
                elif c_max < 65535:
                    optimized[col] = optimized[col].astype(np.uint16)
                elif c_max < 4294967295:
                    optimized[col] = optimized[col].astype(np.uint32)
            else:
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    optimized[col] = optimized[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    optimized[col] = optimized[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    optimized[col] = optimized[col].astype(np.int32)

        elif col_type == 'float64':
            optimized[col] = optimized[col].astype(np.float32)

    return optimized


def convert_csv_to_parquet(csv_path: str, parquet_path: str, chunk_size: int = nrows_per_chunk) -> bool:
    """
    Convert a single CSV to Parquet with memory-efficient chunked processing.
    Returns True on success, False on failure.
    """
    try:
        # Check if already processed
        if ckpt.is_csv_processed(os.path.basename(csv_path)) and os.path.exists(parquet_path):
            print(f"  [SKIP] Already processed: {os.path.basename(csv_path)}")
            return True

        print(f"  Processing: {os.path.basename(csv_path)}")
        start_time = time.time()

        if POLARS_AVAILABLE:
            # Polars lazy streaming for large files
            try:
                # Try reading entire file first (if small enough)
                df = pl.read_csv(csv_path, infer_schema_length=10000, low_memory=True)
                df = optimize_dtypes_polars(df)
                df.write_parquet(parquet_path, compression='zstd')
            except Exception:
                # Fallback to chunked reading for very large files
                chunks = []
                for chunk in pd.read_csv(csv_path, chunksize=chunk_size, low_memory=False):
                    chunk = optimize_dtypes_pandas(chunk)
                    chunks.append(pl.from_pandas(chunk))
                    gc.collect()

                if chunks:
                    df = pl.concat(chunks)
                    df = optimize_dtypes_polars(df)
                    df.write_parquet(parquet_path, compression='zstd')
        else:
            # Pandas chunked reading
            writer = None
            for i, chunk in enumerate(pd.read_csv(csv_path, chunksize=chunk_size, low_memory=False)):
                chunk = optimize_dtypes_pandas(chunk)
                table = pa.Table.from_pandas(chunk)

                if writer is None:
                    writer = pq.ParquetWriter(parquet_path, table.schema, compression='zstd')
                writer.write_table(table)

                del chunk, table
                gc.collect()

            if writer:
                writer.close()

        elapsed = time.time() - start_time
        file_size_mb = os.path.getsize(parquet_path) / (1024**2)
        print(f"    Done in {elapsed:.1f}s | Parquet size: {file_size_mb:.1f} MB")

        # Mark as processed
        ckpt.mark_csv_processed(os.path.basename(csv_path), parquet_path)
        mem_monitor.log(force=True)

        return True

    except Exception as e:
        print(f"    ERROR: {str(e)}")
        logger.log_event("conversion_error", {'file': csv_path, 'error': str(e)})

        # Cleanup partial file
        if os.path.exists(parquet_path):
            os.remove(parquet_path)

        return False


def build_parquet_pipeline(dataset_info: Dict) -> List[str]:
    """
    Build industrial-grade CSV to Parquet conversion pipeline.
    Process one file at a time with progress tracking and recovery.
    """
    print("\n" + "="*60)
    print("CSV -> PARQUET CONVERSION PIPELINE")
    print("="*60)

    os.makedirs(PARQUET_PATH, exist_ok=True)

    csv_files = dataset_info['csv_files']
    data_path = dataset_info['data_path']

    successful = []
    failed = []

    for i, csv_file in enumerate(csv_files, 1):
        csv_path = os.path.join(data_path, csv_file)
        parquet_file = csv_file.replace('.csv', '.parquet')
        parquet_path = os.path.join(PARQUET_PATH, parquet_file)

        print(f"\n[{i}/{len(csv_files)}] Converting {csv_file}")

        success = convert_csv_to_parquet(csv_path, parquet_path)
        if success:
            successful.append(parquet_path)
        else:
            failed.append(csv_file)

        # Periodic memory cleanup
        if i % 5 == 0:
            gc.collect()
            mem_monitor.log(force=True)

    # Summary
    total_parquet_size = sum(os.path.getsize(p) for p in successful) / (1024**3)
    print(f"\n{'='*60}")
    print(f"CONVERSION SUMMARY")
    print(f"  Successful: {len(successful)}/{len(csv_files)}")
    print(f"  Failed: {len(failed)}")
    if failed:
        print(f"  Failed files: {failed}")
    print(f"  Total Parquet size: {total_parquet_size:.2f} GB")
    if total_parquet_size > 0:
        print(f"  Compression ratio: {dataset_info['total_size_gb']/total_parquet_size:.2f}x")
    print("="*60)

    logger.log_param("parquet_files_created", len(successful))
    logger.log_param("parquet_total_size_gb", round(total_parquet_size, 2))
    logger.log_param("conversion_failed", len(failed))

    return successful

# Execute conversion (or skip if already done)
if ckpt.is_step_complete('parquet_conversion'):
    print("\n[INFO] Parquet conversion already complete. Loading existing files...")
    parquet_files = ckpt.get_processed_parquets()
else:
    parquet_files = build_parquet_pipeline(dataset_info)
    if parquet_files:
        ckpt.mark_step_complete('parquet_conversion')

mem_monitor.log(force=True)


CSV -> PARQUET CONVERSION PIPELINE

[1/74] Converting data_1.csv
  Processing: data_1.csv
    Done in 4.7s | Parquet size: 19.5 MB
2026-05-11 20:05:32,377 [INFO] [EVENT] checkpoint_saved | {"state": {"csv_files_processed": ["data_1.csv"], "parquet_files_created": ["/kaggle/working/parquet_data/data_1.parquet"], "last_successful_step": null, "feature_engineering_done": false, "train_test_split_done": false, "model_trained": false, "timestamp": "2026-05-11T20:05:32.376907"}}
2026-05-11 20:05:32,378 [INFO] [EVENT] memory_status | {"ram_used_gb": 1.37, "ram_available_gb": 29.92, "process_rss_gb": 1.11, "peak_ram_gb": 1.11, "ram_percent": 4.6}

[2/74] Converting data_2.csv
  Processing: data_2.csv
    Done in 9.0s | Parquet size: 19.2 MB
2026-05-11 20:05:41,334 [INFO] [EVENT] checkpoint_saved | {"state": {"csv_files_processed": ["data_1.csv", "data_2.csv"], "parquet_files_created": ["/kaggle/working/parquet_data/data_1.parquet", "/kaggle/working/parquet_data/data_2.parquet"], "last_success

{'total_gb': 31.350616455078125,
 'available_gb': 30.07219696044922,
 'used_gb': 1.5261497497558594,
 'process_rss_gb': 1.2259101867675781,
 'process_vms_gb': 5.008358001708984,
 'percent': 4.1}

In [6]:
# ==============================================================================
# SECTION 4 — SCALABLE EXPLORATORY DATA ANALYSIS
# ==============================================================================

# This cell should RIGHT BEFORE Section 4 (or at the top of Section 4)
# It ensures safe_plotly_export is available even when running cells individually

import os
import plotly.graph_objects as go

def safe_plotly_export(fig, html_path, png_path=None, scale=2):
    """Export Plotly figure to HTML always, PNG only if kaleido available."""
    try:
        fig.write_html(html_path)
    except Exception as e:
        print(f"  [WARN] Failed to write HTML: {e}")
    if png_path:
        try:
            fig.write_image(png_path, scale=scale)
        except Exception as e:
            if "kaleido" in str(e).lower():
                print(f"  [WARN] kaleido missing - skipping PNG: {os.path.basename(png_path)}")
            else:
                print(f"  [WARN] PNG export failed: {e}")


class ScalableEDA:
    """
    Memory-efficient EDA for massive datasets.
    Uses sampling and lazy evaluation. Never loads all data simultaneously.
    """

    def __init__(self, parquet_files: List[str]):
        self.parquet_files = parquet_files
        self.sample_cache = None
        self.aggregated_stats = {}

    def _load_sample(self, fraction: float = 0.01, max_rows: int = 100_000) -> pd.DataFrame:
        """Load a stratified sample for visualization."""
        if self.sample_cache is not None:
            return self.sample_cache

        samples = []
        rows_collected = 0

        for pf in self.parquet_files:
            if rows_collected >= max_rows:
                break

            if POLARS_AVAILABLE:
                df = pl.read_parquet(pf)
                n_rows = len(df)
                sample_n = min(int(n_rows * fraction), max_rows - rows_collected)
                if sample_n > 0:
                    sample = df.sample(n=sample_n, seed=RANDOM_SEED)
                    samples.append(sample.to_pandas())
                    rows_collected += sample_n
                del df
            else:
                df = pd.read_parquet(pf)
                n_rows = len(df)
                sample_n = min(int(n_rows * fraction), max_rows - rows_collected)
                if sample_n > 0:
                    samples.append(df.sample(n=sample_n, random_state=RANDOM_SEED))
                    rows_collected += sample_n
                del df

            gc.collect()

        if samples:
            self.sample_cache = pd.concat(samples, ignore_index=True)
        else:
            self.sample_cache = pd.DataFrame()

        return self.sample_cache

    def analyze_class_distribution(self) -> Dict:
        """Analyze target class distribution across all files."""
        print("\n[EDA] Analyzing class distribution...")

        label_counts = defaultdict(int)
        total_rows = 0
        label_col_found = None

        # Possible label column names in Bot-IoT dataset
        possible_labels = ['label', 'Label', 'class', 'Class', 'attack', 'Attack', 
                          'category', 'Category', 'type', 'Type', 'subcategory']

        for pf in self.parquet_files:
            try:
                if POLARS_AVAILABLE:
                    df = pl.read_parquet(pf)

                    # Find the label column
                    if label_col_found is None:
                        for lbl in possible_labels:
                            if lbl in df.columns:
                                label_col_found = lbl
                                print(f"  [INFO] Found label column: '{lbl}'")
                                break

                    if label_col_found and label_col_found in df.columns:
                        counts = df[label_col_found].value_counts()
                        for row in counts.iter_rows():
                            label_counts[row[0]] += row[1]
                        total_rows += len(df)
                    else:
                        print(f"  [WARN] No label column found in {os.path.basename(pf)}")
                        total_rows += len(df)

                    del df
                else:
                    df = pd.read_parquet(pf)

                    if label_col_found is None:
                        for lbl in possible_labels:
                            if lbl in df.columns:
                                label_col_found = lbl
                                print(f"  [INFO] Found label column: '{lbl}'")
                                break

                    if label_col_found and label_col_found in df.columns:
                        vc = df[label_col_found].value_counts()
                        for idx, val in vc.items():
                            label_counts[idx] += val
                        total_rows += len(df)
                    else:
                        print(f"  [WARN] No label column found in {os.path.basename(pf)}")
                        total_rows += len(df)

                    del df
            except Exception as e:
                print(f"  [WARN] Error reading {os.path.basename(pf)}: {e}")
                continue

            gc.collect()

        # Convert to proper format
        distribution = dict(label_counts)

        print(f"  Total rows analyzed: {total_rows:,}")
        print(f"  Class distribution: {distribution}")

        # Plotly interactive class distribution
        fig = go.Figure(data=[
            go.Bar(
                x=list(distribution.keys()),
                y=list(distribution.values()),
                marker_color=['#2ecc71' if 'benign' in str(k).lower() or 'normal' in str(k).lower() else '#e74c3c' for k in distribution.keys()],
                text=[f"{v:,} ({v/total_rows*100:.1f}%)" for v in distribution.values()],
                textposition='auto'
            )
        ])
        fig.update_layout(
            title='Bot-IoT Class Distribution (Full Dataset)',
            xaxis_title='Class',
            yaxis_title='Count',
            template='plotly_white',
            height=500
        )
        safe_plotly_export(fig, 
                           os.path.join(WORKING_PATH, 'class_distribution.html'),
                           os.path.join(WORKING_PATH, 'class_distribution.png'))

        logger.log_param("total_rows", total_rows)
        logger.log_param("num_classes", len(distribution))

        return {
            'distribution': distribution,
            'total_rows': total_rows,
            'imbalance_ratio': max(distribution.values()) / min(distribution.values()) if distribution else 1
        }

    def analyze_missing_values(self) -> pd.DataFrame:
        """Analyze missing values using sampled data."""
        print("[EDA] Analyzing missing values...")
        sample = self._load_sample(fraction=0.005)

        if sample.empty:
            return pd.DataFrame()

        missing = sample.isnull().sum()
        missing_pct = (missing / len(sample) * 100).round(2)

        missing_df = pd.DataFrame({
            'column': missing.index,
            'missing_count': missing.values,
            'missing_pct': missing_pct.values
        }).sort_values('missing_pct', ascending=False)

        print(f"  Columns with missing values: {(missing > 0).sum()}")

        # Plot missing values heatmap
        if (missing > 0).any():
            plt.figure(figsize=(12, 8))
            missing_cols = missing_df[missing_df['missing_pct'] > 0]
            sns.barplot(data=missing_cols, x='missing_pct', y='column', palette='Reds_r')
            plt.title('Missing Values by Column (Sample)', fontsize=14)
            plt.xlabel('Missing Percentage (%)')
            plt.tight_layout()
            plt.savefig(os.path.join(WORKING_PATH, 'missing_values.png'), dpi=150, bbox_inches='tight')
            plt.close()

        return missing_df

    def analyze_feature_cardinality(self) -> pd.DataFrame:
        """Analyze feature cardinality for categorical encoding decisions."""
        print("[EDA] Analyzing feature cardinality...")
        sample = self._load_sample(fraction=0.01)

        if sample.empty:
            return pd.DataFrame()

        cardinality = []
        for col in sample.columns:
            n_unique = sample[col].nunique()
            cardinality.append({
                'feature': col,
                'n_unique': n_unique,
                'cardinality_ratio': n_unique / len(sample),
                'dtype': str(sample[col].dtype),
                'suggested_encoding': 'categorical' if n_unique < 100 else 'numerical'
            })

        card_df = pd.DataFrame(cardinality).sort_values('n_unique', ascending=False)
        print(f"  Features analyzed: {len(card_df)}")

        return card_df

    def analyze_correlations(self, top_n: int = 20) -> pd.DataFrame:
        """Correlation analysis on numerical features (sampled)."""
        print("[EDA] Analyzing correlations...")
        sample = self._load_sample(fraction=0.01)

        if sample.empty:
            return pd.DataFrame()

        # Select numerical columns
        num_cols = sample.select_dtypes(include=[np.number]).columns.tolist()
        if len(num_cols) > 50:
            # Sample columns if too many
            num_cols = np.random.choice(num_cols, 50, replace=False).tolist()

        corr_matrix = sample[num_cols].corr()

        # Plot correlation heatmap
        plt.figure(figsize=(14, 12))
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
        sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='RdBu_r', center=0,
                   square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
        plt.title('Feature Correlation Matrix (Sample)', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(WORKING_PATH, 'correlation_heatmap.png'), dpi=150, bbox_inches='tight')
        plt.close()

        # Find highly correlated pairs
        corr_pairs = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i+1, len(corr_matrix.columns)):
                val = corr_matrix.iloc[i, j]
                if abs(val) > 0.8:
                    corr_pairs.append({
                        'feature_1': corr_matrix.columns[i],
                        'feature_2': corr_matrix.columns[j],
                        'correlation': val
                    })

        corr_pairs_df = pd.DataFrame(corr_pairs).sort_values('correlation', key=abs, ascending=False)
        print(f"  Highly correlated pairs (|r| > 0.8): {len(corr_pairs_df)}")

        return corr_pairs_df

    def analyze_duplicates(self) -> Dict:
        """Detect duplicate and near-duplicate flows."""
        print("[EDA] Analyzing duplicates...")
        sample = self._load_sample(fraction=0.005, max_rows=50_000)

        if sample.empty or len(sample) < 100:
            return {}

        # Exact duplicates
        exact_dups = sample.duplicated().sum()
        exact_dup_pct = exact_dups / len(sample) * 100

        # Near-duplicates (same key flow features, slight differences in counts)
        flow_key_cols = [c for c in sample.columns if any(x in c.lower() for x in 
                        ['src', 'dst', 'sport', 'dport', 'proto', 'ip'])]

        near_dups = 0
        if len(flow_key_cols) >= 3:
            near_dups = sample.duplicated(subset=flow_key_cols[:5]).sum() - exact_dups

        print(f"  Exact duplicates: {exact_dups:,} ({exact_dup_pct:.2f}%)")
        print(f"  Near-duplicates: {near_dups:,}")

        return {
            'exact_duplicates': exact_dups,
            'exact_duplicate_pct': exact_dup_pct,
            'near_duplicates': near_dups,
            'flow_key_columns': flow_key_cols
        }

    def run_full_eda(self) -> Dict:
        """Execute complete EDA pipeline."""
        print("\n" + "="*60)
        print("SCALABLE EXPLORATORY DATA ANALYSIS")
        print("="*60)

        results = {}
        results['class_distribution'] = self.analyze_class_distribution()
        results['missing_values'] = self.analyze_missing_values()
        results['feature_cardinality'] = self.analyze_feature_cardinality()
        results['correlations'] = self.analyze_correlations()
        results['duplicates'] = self.analyze_duplicates()

        print("\n[EDA] Complete. Visualizations saved to working directory.")
        print("="*60)

        logger.log_event("eda_complete", {
            'total_rows': results['class_distribution'].get('total_rows', 0),
            'num_classes': results['class_distribution'].get('num_classes', 0),
            'imbalance_ratio': results['class_distribution'].get('imbalance_ratio', 1)
        })

        return results

# Run EDA
eda = ScalableEDA(parquet_files)
eda_results = eda.run_full_eda()
mem_monitor.log(force=True)



SCALABLE EXPLORATORY DATA ANALYSIS

[EDA] Analyzing class distribution...
  [INFO] Found label column: 'attack'
  Total rows analyzed: 72,370,443
  Class distribution: {1: 72361002, 0: 9441}
  [WARN] kaleido missing - skipping PNG: class_distribution.png
2026-05-11 20:13:25,688 [INFO] [PARAM] total_rows = 72370443
2026-05-11 20:13:25,689 [INFO] [PARAM] num_classes = 2
[EDA] Analyzing missing values...
  Columns with missing values: 8
[EDA] Analyzing feature cardinality...
  Features analyzed: 35
[EDA] Analyzing correlations...
  Highly correlated pairs (|r| > 0.8): 11
[EDA] Analyzing duplicates...
  Exact duplicates: 0 (0.00%)
  Near-duplicates: 25,107

[EDA] Complete. Visualizations saved to working directory.
2026-05-11 20:13:32,693 [INFO] [EVENT] eda_complete | {"total_rows": 72370443, "num_classes": 0, "imbalance_ratio": 7664.548458849698}
2026-05-11 20:13:32,695 [INFO] [EVENT] memory_status | {"ram_used_gb": 1.82, "ram_available_gb": 30.06, "process_rss_gb": 1.56, "peak_ram_gb": 

{'total_gb': 31.350616455078125,
 'available_gb': 30.056293487548828,
 'used_gb': 1.822906494140625,
 'process_rss_gb': 1.5611801147460938,
 'process_vms_gb': 6.318950653076172,
 'percent': 4.1}

In [7]:
# ==============================================================================
# SECTION 5 — ADVANCED FEATURE ENGINEERING
# ==============================================================================

class FeatureEngineer:
    """
    Production-grade feature engineering for cybersecurity data.
    Safe feature generation with null protection and divide-by-zero guards.
    """

    def __init__(self):
        self.feature_list = []
        self.scaler = None

    @staticmethod
    def safe_divide(numerator, denominator, fill_value=0.0):
        """Safe division with zero protection."""
        return np.where(
            (denominator != 0) & (~np.isnan(denominator)) & (~np.isnan(numerator)),
            numerator / denominator,
            fill_value
        )

    def create_cybersecurity_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Create advanced cybersecurity-specific features.
        Includes packet/byte ratios, session intensity, entropy-like features.
        """
        df = df.copy()
        new_features = []

        # Identify common column patterns
        pkt_cols = [c for c in df.columns if 'pkt' in c.lower() or 'packet' in c.lower()]
        byte_cols = [c for c in df.columns if 'byte' in c.lower() or 'byt' in c.lower()]
        dur_cols = [c for c in df.columns if 'dur' in c.lower() or 'time' in c.lower()]
        proto_cols = [c for c in df.columns if 'proto' in c.lower()]

        # 1. Packet/Byte Ratios
        for pkt_col in pkt_cols[:2]:  # Limit to avoid explosion
            for byte_col in byte_cols[:2]:
                if pkt_col in df.columns and byte_col in df.columns:
                    feat_name = f"{pkt_col}_per_{byte_col}"
                    df[feat_name] = self.safe_divide(df[pkt_col].astype(float), df[byte_col].astype(float))
                    new_features.append(feat_name)

        # 2. Bytes per packet (aggregate)
        if byte_cols and pkt_cols:
            total_bytes = df[byte_cols[0]].astype(float) if byte_cols[0] in df.columns else pd.Series(0, index=df.index)
            total_pkts = df[pkt_cols[0]].astype(float) if pkt_cols[0] in df.columns else pd.Series(1, index=df.index)
            df['bytes_per_packet'] = self.safe_divide(total_bytes, total_pkts)
            new_features.append('bytes_per_packet')

        # 3. Session intensity (packets per duration)
        if dur_cols and pkt_cols:
            dur_col = dur_cols[0]
            pkt_col = pkt_cols[0]
            if dur_col in df.columns and pkt_col in df.columns:
                df['session_intensity'] = self.safe_divide(
                    df[pkt_col].astype(float), 
                    df[dur_col].astype(float).replace(0, np.nan)
                )
                new_features.append('session_intensity')

        # 4. Duration transformations
        for dur_col in dur_cols[:1]:
            if dur_col in df.columns:
                dur = df[dur_col].astype(float).replace(0, np.nan)
                df[f'{dur_col}_log'] = np.log1p(dur.fillna(0))
                df[f'{dur_col}_sqrt'] = np.sqrt(dur.fillna(0))
                new_features.extend([f'{dur_col}_log', f'{dur_col}_sqrt'])

        # 5. Connection density (if IP/port info available)
        ip_cols = [c for c in df.columns if 'ip' in c.lower()]
        port_cols = [c for c in df.columns if 'port' in c.lower()]

        if len(ip_cols) >= 2 and len(port_cols) >= 2:
            # Unique destination ratio
            src_ip, dst_ip = ip_cols[0], ip_cols[1]
            src_port, dst_port = port_cols[0], port_cols[1]

            # Create interaction features
            df['src_dst_interaction'] = (
                df[src_ip].astype(str) + '_' + df[dst_ip].astype(str) + '_' + 
                df[src_port].astype(str) + '_' + df[dst_port].astype(str)
            ).astype('category').cat.codes
            new_features.append('src_dst_interaction')

        # 6. Entropy-like features (for payload/flow randomness)
        if byte_cols:
            byte_col = byte_cols[0]
            if byte_col in df.columns:
                # Approximate entropy using coefficient of variation
                mean_bytes = df[byte_col].astype(float).mean()
                std_bytes = df[byte_col].astype(float).std()
                df['byte_entropy_proxy'] = self.safe_divide(std_bytes, mean_bytes)
                new_features.append('byte_entropy_proxy')

        # 7. Protocol encoding (if categorical)
        for proto_col in proto_cols[:1]:
            if proto_col in df.columns:
                if df[proto_col].dtype == 'object' or df[proto_col].dtype.name == 'category':
                    df[f'{proto_col}_encoded'] = df[proto_col].astype('category').cat.codes
                    new_features.append(f'{proto_col}_encoded')

        # 8. Temporal features (if timestamp exists)
        time_cols = [c for c in df.columns if any(x in c.lower() for x in ['time', 'ts', 'date', 'stamp'])]
        for time_col in time_cols[:1]:
            if time_col in df.columns:
                try:
                    ts = pd.to_datetime(df[time_col], errors='coerce', unit='s')
                    df['hour_of_day'] = ts.dt.hour.fillna(-1)
                    df['day_of_week'] = ts.dt.dayofweek.fillna(-1)
                    new_features.extend(['hour_of_day', 'day_of_week'])
                except:
                    pass

        self.feature_list = list(df.columns)

        print(f"[FE] Created {len(new_features)} new features")
        print(f"[FE] Total features: {len(self.feature_list)}")

        # Handle infinities and NaNs
        df = df.replace([np.inf, -np.inf], np.nan)

        logger.log_param("original_features", len(df.columns) - len(new_features))
        logger.log_param("engineered_features", len(new_features))
        logger.log_param("total_features", len(self.feature_list))

        return df

    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Fit and transform features."""
        return self.create_cybersecurity_features(df)

In [8]:
# ==============================================================================
# SECTION 6 — DATA REDUCTION & STRATIFIED SAMPLING (MEMORY-SAFE)
# ==============================================================================

def create_stratified_subset(
    parquet_files: List[str],
    target_per_class: int = None,
    label_col: str = 'attack',
    max_rows_per_file: int = 500_000
) -> pd.DataFrame:
    """
    Create a GLOBALLY balanced subset using MEMORY-EFFICIENT streaming.
    CRITICAL: Processes files one at a time, never keeps all data in memory.
    """
    print("\n" + "="*60)
    print("DATA REDUCTION & STRATIFIED SAMPLING (MEMORY-SAFE)")
    print("="*60)

    print(f"[INFO] Max rows per file: {max_rows_per_file:,}")
    print(f"[INFO] Total files: {len(parquet_files)}")
    if target_per_class:
        print(f"[INFO] Target per class: {target_per_class:,}")
    else:
        print("[INFO] Mode: USE ALL AVAILABLE BENIGN + MATCHED ATTACK")

    # PHASE 1: Light-weight pass to count benign per file
    print("\n[PHASE 1] Counting benign samples per file...")

    file_stats = []
    total_benign = 0
    total_attack = 0

    for i, pf in enumerate(parquet_files):
        # Read only the label column to save memory
        if POLARS_AVAILABLE:
            df_small = pl.read_parquet(pf, columns=[label_col])
            n_benign = len(df_small.filter(pl.col(label_col) == 0))
            n_attack = len(df_small.filter(pl.col(label_col) == 1))
        else:
            df_small = pd.read_parquet(pf, columns=[label_col])
            n_benign = (df_small[label_col] == 0).sum()
            n_attack = (df_small[label_col] == 1).sum()

        total_benign += int(n_benign)
        total_attack += int(n_attack)
        file_stats.append({'file': pf, 'benign': int(n_benign), 'attack': int(n_attack)})

        del df_small

        if (i + 1) % 10 == 0:
            print(f"  Processed {i+1}/{len(parquet_files)} files | Benign so far: {total_benign:,}")

    print(f"\n[INFO] Total benign across all files: {total_benign:,}")
    print(f"[INFO] Total attack across all files: {total_attack:,}")

    if total_benign == 0:
        raise ValueError("No benign samples found in any file!")

    # PHASE 2: Determine target and sampling strategy
    if target_per_class is None:
        target = total_benign
        print(f"[INFO] Will collect ALL {total_benign:,} benign + {total_benign:,} matched attack")
    else:
        target = min(total_benign, total_attack, target_per_class)
        print(f"[INFO] Target per class: {target:,}")

    # PHASE 3: Stream sampling — process each file, keep only what's needed
    print("\n[PHASE 2] Streaming samples from files...")

    benign_collected = []
    attack_collected = []
    benign_so_far = 0
    attack_so_far = 0

    for i, stats in enumerate(file_stats):
        pf = stats['file']
        n_benign_file = stats['benign']
        n_attack_file = stats['attack']

        # Calculate how many to take from this file
        # Proportional to file's contribution, but cap at available
        if target_per_class is None:
            # Take ALL benign from each file
            benign_to_take = n_benign_file
            attack_to_take = min(n_attack_file, n_benign_file)  # Match 1:1 per file
        else:
            # Proportional sampling
            benign_to_take = min(n_benign_file, max(1, int(target * n_benign_file / total_benign)))
            attack_to_take = min(n_attack_file, max(1, int(target * n_attack_file / total_attack)))

        # Skip if nothing to take
        if benign_to_take == 0 and attack_to_take == 0:
            continue

        # Read full file only if we need samples from it
        if POLARS_AVAILABLE:
            df = pl.read_parquet(pf)

            # Cast types to avoid issues
            for col in df.columns:
                dtype = df[col].dtype
                if dtype == pl.Categorical:
                    df = df.with_columns(pl.col(col).cast(pl.Utf8))
                elif dtype == pl.Float32:
                    df = df.with_columns(pl.col(col).cast(pl.Float64))
        else:
            df = pd.read_parquet(pf)
            df = pl.from_pandas(df)

        # Sample from this file
        if benign_to_take > 0 and n_benign_file > 0:
            benign_df = df.filter(pl.col(label_col) == 0)
            if len(benign_df) > 0:
                sample_size = min(benign_to_take, len(benign_df))
                benign_sample = benign_df.sample(n=sample_size, seed=RANDOM_SEED + i)
                benign_collected.append(benign_sample.to_pandas())
                benign_so_far += sample_size

        if attack_to_take > 0 and n_attack_file > 0:
            attack_df = df.filter(pl.col(label_col) == 1)
            if len(attack_df) > 0:
                sample_size = min(attack_to_take, len(attack_df))
                attack_sample = attack_df.sample(n=sample_size, seed=RANDOM_SEED + i)
                attack_collected.append(attack_sample.to_pandas())
                attack_so_far += sample_size

        del df
        gc.collect()

        if (i + 1) % 10 == 0 or i == len(file_stats) - 1:
            print(f"  [File {i+1}] Benign: {benign_so_far:,}/{target} | Attack: {attack_so_far:,}/{target}")
            mem_monitor.log(force=True)

        # Early stop if we have enough
        if target_per_class and benign_so_far >= target and attack_so_far >= target:
            print(f"\n[INFO] Early stop: reached target of {target} per class")
            break

    # PHASE 4: Combine collected samples
    print("\n[INFO] Combining collected samples...")

    if not benign_collected or not attack_collected:
        raise ValueError("No samples collected")

    all_benign = pd.concat(benign_collected, ignore_index=True)
    all_attack = pd.concat(attack_collected, ignore_index=True)

    # Trim to exact target if needed
    if len(all_benign) > target:
        all_benign = all_benign.sample(n=target, random_state=RANDOM_SEED)
    if len(all_attack) > target:
        all_attack = all_attack.sample(n=target, random_state=RANDOM_SEED)

    # Combine and shuffle
    combined = pd.concat([all_benign, all_attack], ignore_index=True)
    combined = combined.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

    print(f"\n[SAMPLE] Combined dataset shape: {combined.shape}")
    print(f"[SAMPLE] Total rows: {len(combined):,}")

    if label_col in combined.columns:
        class_counts = combined[label_col].value_counts().sort_index()
        print(f"[SAMPLE] Class distribution:")
        print(class_counts)

        if len(class_counts) == 2 and abs(class_counts.iloc[0] - class_counts.iloc[1]) <= 1:
            print("[SAMPLE] ✓ Dataset is BALANCED")
        else:
            print("[SAMPLE] ✗ WARNING: Dataset is NOT balanced!")

    logger.log_param("sampled_rows", len(combined))
    logger.log_param("sampled_features", combined.shape[1])
    logger.log_param("benign_samples", len(all_benign))
    logger.log_param("attack_samples", len(all_attack))

    return combined

# Execute sampling
if not ckpt.is_step_complete('data_sampling'):
    df_sample = create_stratified_subset(
        parquet_files, 
        target_per_class=None
    )

    fe = FeatureEngineer()
    df_processed = fe.fit_transform(df_sample)

    for col in df_processed.columns:
        if df_processed[col].dtype == 'object':
            df_processed[col] = df_processed[col].astype(str)

    cache_path = os.path.join(CHECKPOINT_PATH, 'processed_sample.parquet')
    df_processed.to_parquet(cache_path, compression='zstd')
    ckpt.mark_step_complete('data_sampling')

    del df_sample
    gc.collect()
else:
    print("\n[INFO] Loading cached processed sample...")
    cache_path = os.path.join(CHECKPOINT_PATH, 'processed_sample.parquet')
    df_processed = pd.read_parquet(cache_path)
    fe = FeatureEngineer()
    fe.feature_list = list(df_processed.columns)

mem_monitor.log(force=True)


DATA REDUCTION & STRATIFIED SAMPLING (MEMORY-SAFE)
[INFO] Max rows per file: 500,000
[INFO] Total files: 73
[INFO] Mode: USE ALL AVAILABLE BENIGN + MATCHED ATTACK

[PHASE 1] Counting benign samples per file...
  Processed 10/73 files | Benign so far: 7,384
  Processed 20/73 files | Benign so far: 7,722
  Processed 30/73 files | Benign so far: 8,067
  Processed 40/73 files | Benign so far: 8,381
  Processed 50/73 files | Benign so far: 8,709
  Processed 60/73 files | Benign so far: 9,026
  Processed 70/73 files | Benign so far: 9,355

[INFO] Total benign across all files: 9,441
[INFO] Total attack across all files: 72,361,002
[INFO] Will collect ALL 9,441 benign + 9,441 matched attack

[PHASE 2] Streaming samples from files...
  [File 10] Benign: 7,384/9441 | Attack: 7,384/9441
2026-05-11 20:13:36,934 [INFO] [EVENT] memory_status | {"ram_used_gb": 2.51, "ram_available_gb": 29.95, "process_rss_gb": 2.25, "peak_ram_gb": 2.25, "ram_percent": 4.5}
  [File 20] Benign: 7,722/9441 | Attack: 7

{'total_gb': 31.350616455078125,
 'available_gb': 30.020984649658203,
 'used_gb': 2.5766677856445312,
 'process_rss_gb': 2.3033599853515625,
 'process_vms_gb': 7.229122161865234,
 'percent': 4.2}

In [9]:
# ==============================================================================
# SECTION 7 — TRAIN/VALIDATION/TEST SPLIT
# ==============================================================================

from sklearn.model_selection import train_test_split

def create_balanced_splits(df_processed, label_col='attack', val_size=0.15, test_size=0.15, random_state=42):
    """
    Create train/val/test splits where VALIDATION and TEST are BALANCED.
    Training set uses ALL data with class weights (handled in Section 8).

    CRITICAL FIX: Ensures val/test have MEANINGFUL size (not 142 samples).
    For extreme imbalance (e.g., 334 benign in 2.5M rows):
    - Use ALL benign for val/test (reserve 30%)
    - Match with EQUAL attack samples
    - Training gets remaining 70% benign + ALL remaining attack
    """
    print("\n" + "="*60)
    print("TRAIN/VALIDATION/TEST SPLIT (BALANCED)")
    print("="*60)

    X = df_processed.drop(columns=[label_col])
    y = df_processed[label_col]

    print(f"[INFO] Total samples: {len(y):,}")
    print(f"[INFO] Original class distribution:")
    print(y.value_counts().sort_index())

    # Get indices by class
    benign_indices = np.where(y == 0)[0]
    attack_indices = np.where(y == 1)[0]

    n_benign = len(benign_indices)
    n_attack = len(attack_indices)

    print(f"[INFO] Benign: {n_benign:,} | Attack: {n_attack:,}")

    # NEW FIX: For extreme imbalance, use larger val/test by:
    # 1. Using ALL benign samples (don't waste them)
    # 2. Matching with equal attack samples
    # 3. If still small, use more attack for val/test while keeping balance

    # Reserve 30% of benign for val+test
    n_benign_val = int(n_benign * 0.25)   # At least 25%
    n_benign_test = int(n_benign * 0.25)  # At least 25%
    n_benign_train = n_benign - n_benign_val - n_benign_test  # 50%

    # Ensure train has some benign
    if n_benign_train < 50:
        # Redistribute: give train 50%, val 25%, test 25%
        n_benign_train = max(n_benign // 2, 50)
        remaining = n_benign - n_benign_train
        n_benign_val = remaining // 2
        n_benign_test = remaining - n_benign_val

    print(f"\n[INFO] Benign split: Train={n_benign_train}, Val={n_benign_val}, Test={n_benign_test}")

    # Shuffle and split benign indices
    np.random.seed(random_state)
    np.random.shuffle(benign_indices)

    benign_train_idx = benign_indices[:n_benign_train]
    benign_val_idx = benign_indices[n_benign_train:n_benign_train + n_benign_val]
    benign_test_idx = benign_indices[n_benign_train + n_benign_val:]

    # For attack: match benign amounts for val/test, rest for train
    np.random.shuffle(attack_indices)

    # Use EQUAL attack samples for val/test to maintain balance
    # But also ensure minimum size for statistical significance
    MIN_VAL_TEST_SIZE = 1000  # NEW: Minimum samples per class in val/test

    attack_val_count = max(n_benign_val, MIN_VAL_TEST_SIZE)
    attack_test_count = max(n_benign_test, MIN_VAL_TEST_SIZE)

    # Cap at available attack samples
    attack_val_count = min(attack_val_count, n_attack // 3)
    attack_test_count = min(attack_test_count, n_attack // 3)

    attack_val_idx = attack_indices[:attack_val_count]
    attack_test_idx = attack_indices[attack_val_count:attack_val_count + attack_test_count]
    attack_train_idx = attack_indices[attack_val_count + attack_test_count:]

    print(f"[INFO] Attack split: Train={len(attack_train_idx)}, Val={len(attack_val_idx)}, Test={len(attack_test_idx)}")

    # Combine indices
    train_idx = np.concatenate([benign_train_idx, attack_train_idx])
    val_idx = np.concatenate([benign_val_idx, attack_val_idx])
    test_idx = np.concatenate([benign_test_idx, attack_test_idx])

    # Shuffle each split
    np.random.shuffle(train_idx)
    np.random.shuffle(val_idx)
    np.random.shuffle(test_idx)

    # Create splits
    X_train = X.iloc[train_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)

    X_val = X.iloc[val_idx].reset_index(drop=True)
    y_val = y.iloc[val_idx].reset_index(drop=True)

    X_test = X.iloc[test_idx].reset_index(drop=True)
    y_test = y.iloc[test_idx].reset_index(drop=True)

    print(f"\n[SPLIT] Final shapes:")
    print(f"  X_train: {X_train.shape} | y_train: {y_train.shape}")
    print(f"  X_val:   {X_val.shape}   | y_val:   {y_val.shape}")
    print(f"  X_test:  {X_test.shape}  | y_test:  {y_test.shape}")

    # Verify distributions
    print(f"\n[SPLIT] Final class distributions:")
    print(f"  Train: {np.bincount(y_train.astype(int))} (imbalanced, will use class weights)")
    print(f"  Val:   {np.bincount(y_val.astype(int))} (balanced)")
    print(f"  Test:  {np.bincount(y_test.astype(int))} (balanced)")

    # Verify minimum size
    val_counts = np.bincount(y_val.astype(int))
    test_counts = np.bincount(y_test.astype(int))

    if min(val_counts) < 100 or min(test_counts) < 100:
        print(f"\n[WARN] Val/Test still small! Consider:")
        print(f"  - Using more data in Section 6 (increase target_per_class)")
        print(f"  - Collecting more benign samples from source data")
    else:
        print(f"\n[INFO] ✓ Val/Test sizes are adequate for reliable evaluation")

    return X_train, X_val, X_test, y_train, y_val, y_test

# Execute split
if not ckpt.is_step_complete('data_split'):
    X_train, X_val, X_test, y_train, y_val, y_test = create_balanced_splits(df_processed)

    # Save splits
    split_data = {
        'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
        'y_train': y_train, 'y_val': y_val, 'y_test': y_test
    }

    for name, data in split_data.items():
        path = os.path.join(CHECKPOINT_PATH, f'{name}.parquet')
        if name.startswith('X_'):
            data.to_parquet(path, compression='zstd')
        else:
            pd.DataFrame({name: data}).to_parquet(path, compression='zstd')

    ckpt.mark_step_complete('data_split')

    del df_processed
    gc.collect()
else:
    print("\n[INFO] Loading cached splits...")
    X_train = pd.read_parquet(os.path.join(CHECKPOINT_PATH, 'X_train.parquet'))
    X_val = pd.read_parquet(os.path.join(CHECKPOINT_PATH, 'X_val.parquet'))
    X_test = pd.read_parquet(os.path.join(CHECKPOINT_PATH, 'X_test.parquet'))
    y_train = pd.read_parquet(os.path.join(CHECKPOINT_PATH, 'y_train.parquet'))['y_train'].values
    y_val = pd.read_parquet(os.path.join(CHECKPOINT_PATH, 'y_val.parquet'))['y_val'].values
    y_test = pd.read_parquet(os.path.join(CHECKPOINT_PATH, 'y_test.parquet'))['y_test'].values

mem_monitor.log(force=True)


TRAIN/VALIDATION/TEST SPLIT (BALANCED)
[INFO] Total samples: 18,882
[INFO] Original class distribution:
attack
0    9441
1    9441
Name: count, dtype: int64
[INFO] Benign: 9,441 | Attack: 9,441

[INFO] Benign split: Train=4721, Val=2360, Test=2360
[INFO] Attack split: Train=4721, Val=2360, Test=2360

[SPLIT] Final shapes:
  X_train: (9442, 46) | y_train: (9442,)
  X_val:   (4720, 46)   | y_val:   (4720,)
  X_test:  (4720, 46)  | y_test:  (4720,)

[SPLIT] Final class distributions:
  Train: [4721 4721] (imbalanced, will use class weights)
  Val:   [2360 2360] (balanced)
  Test:  [2360 2360] (balanced)

[INFO] ✓ Val/Test sizes are adequate for reliable evaluation
2026-05-11 20:14:03,348 [INFO] [EVENT] checkpoint_saved | {"state": {"csv_files_processed": ["data_1.csv", "data_2.csv", "data_3.csv", "data_4.csv", "data_5.csv", "data_6.csv", "data_7.csv", "data_8.csv", "data_9.csv", "data_10.csv", "data_11.csv", "data_12.csv", "data_13.csv", "data_14.csv", "data_15.csv", "data_16.csv", "data

{'total_gb': 31.350616455078125,
 'available_gb': 30.023174285888672,
 'used_gb': 2.5765724182128906,
 'process_rss_gb': 2.30535888671875,
 'process_vms_gb': 7.229122161865234,
 'percent': 4.2}

In [10]:
# ==============================================================================
# SECTION 8 — FEATURE SCALING & CLASS WEIGHTS (FINAL v6 - LEAKAGE FIX)
# ==============================================================================

from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

print("\n" + "="*60)
print("FEATURE SCALING & CLASS WEIGHTS")
print("="*60)

# STEP 0: DROP DATA LEAKAGE COLUMNS
# These columns are derived from the label and would make the task trivial
LEAKAGE_COLUMNS = ['category', 'subcategory', 'subcategory ', 'attack', 'label']

print("[INFO] Checking for data leakage columns...")
for col in LEAKAGE_COLUMNS:
    if col in X_train.columns:
        print(f"  [DROP] Removing leakage column: '{col}'")
        X_train = X_train.drop(columns=[col])
        X_val = X_val.drop(columns=[col])
        X_test = X_test.drop(columns=[col])

# STEP 1: Handle non-numeric columns
print("[INFO] Processing non-numeric columns...")

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
non_numeric_cols = [c for c in X_train.columns if c not in numeric_cols]

print(f"[INFO] Numeric columns: {len(numeric_cols)}")
print(f"[INFO] Non-numeric columns: {non_numeric_cols}")

for col in non_numeric_cols:
    if 'time' in col.lower() or pd.api.types.is_datetime64_any_dtype(X_train[col]):
        print(f"  [CONVERT] {col}: datetime → unix timestamp")
        for df in [X_train, X_val, X_test]:
            df[col] = pd.to_numeric(pd.to_datetime(df[col], errors='coerce')) / 10**9

    elif col in ['sport', 'dport']:
        print(f"  [CONVERT] {col}: string port → numeric")
        for df in [X_train, X_val, X_test]:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    elif X_train[col].nunique() < 50:
        print(f"  [ENCODE] {col}: categorical → label encoded")
        le = LabelEncoder()
        all_values = pd.concat([X_train[col], X_val[col], X_test[col]]).astype(str)
        le.fit(all_values)
        X_train[col] = le.transform(X_train[col].astype(str))
        X_val[col] = le.transform(X_val[col].astype(str))
        X_test[col] = le.transform(X_test[col].astype(str))

    else:
        print(f"  [DROP] {col}: high cardinality string")
        X_train = X_train.drop(columns=[col])
        X_val = X_val.drop(columns=[col])
        X_test = X_test.drop(columns=[col])

# STEP 2: Drop all-NaN / mostly-NaN columns
print("\n[INFO] Checking for all-NaN / mostly-NaN columns...")

cols_to_drop = set()
for df_name, df in [("train", X_train), ("val", X_val), ("test", X_test)]:
    nan_ratio = df.isna().mean()
    all_nan_cols = nan_ratio[nan_ratio == 1.0].index.tolist()
    mostly_nan_cols = nan_ratio[nan_ratio > 0.9].index.tolist()
    cols_to_drop.update(all_nan_cols)
    cols_to_drop.update(mostly_nan_cols)
    if all_nan_cols:
        print(f"  [{df_name}] ALL-NaN: {all_nan_cols}")
    if mostly_nan_cols:
        print(f"  [{df_name}] Mostly-NaN: {mostly_nan_cols}")

if cols_to_drop:
    cols_to_drop = list(cols_to_drop)
    print(f"\n[INFO] Dropping: {cols_to_drop}")
    for df_name, df in [("train", X_train), ("val", X_val), ("test", X_test)]:
        existing = [c for c in cols_to_drop if c in df.columns]
        if existing:
            df.drop(columns=existing, inplace=True)

# STEP 3: Align columns
common_cols = list(set(X_train.columns) & set(X_val.columns) & set(X_test.columns))
X_train = X_train[common_cols]
X_val = X_val[common_cols]
X_test = X_test[common_cols]
print(f"\n[INFO] Common columns: {len(common_cols)}")

# STEP 4: Handle NaN/Inf
print("\n[INFO] Handling NaN/Inf...")
for df_name, df in [("train", X_train), ("val", X_val), ("test", X_test)]:
    nan_count = df.isna().sum().sum()
    if nan_count > 0:
        print(f"  [{df_name}] Filling {nan_count:,} NaN")
        df.fillna(df.median(numeric_only=True), inplace=True)

    numeric_df = df.select_dtypes(include=[np.number])
    inf_mask = np.isinf(numeric_df)
    inf_count = inf_mask.sum().sum()
    if inf_count > 0:
        print(f"  [{df_name}] Replacing {inf_count:,} Inf")
        for col in numeric_df.columns:
            col_inf = inf_mask[col]
            if col_inf.any():
                max_finite = numeric_df.loc[~col_inf, col].max()
                df.loc[col_inf, col] = max_finite

# STEP 5: Verify numeric
remaining_non_numeric = X_train.select_dtypes(exclude=[np.number]).columns.tolist()
if remaining_non_numeric:
    print(f"[WARN] Dropping non-numeric: {remaining_non_numeric}")
    X_train = X_train.drop(columns=remaining_non_numeric)
    X_val = X_val.drop(columns=remaining_non_numeric)
    X_test = X_test.drop(columns=remaining_non_numeric)

print(f"[INFO] Final shapes: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")

# STEP 6: Scale
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
print(f"\n[INFO] Features scaled: {X_train_scaled.shape[1]}")

# STEP 7: Class weights
print("\n[INFO] Computing class weights...")
classes = np.unique(y_train)
class_counts = np.bincount(y_train.astype(int))
print(f"[INFO] Class distribution: {dict(zip(classes, class_counts))}")

weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = {int(c): w for c, w in zip(classes, weights)}
for cls, weight in class_weight_dict.items():
    print(f"  Class {cls}: {weight:.4f}")

n_negative = (y_train == 0).sum()
n_positive = (y_train == 1).sum()
scale_pos_weight = n_positive / n_negative if n_negative > 0 else 1.0
print(f"\n[INFO] scale_pos_weight: {scale_pos_weight:.2f}")

# STEP 8: Save
os.makedirs(MODEL_PATH, exist_ok=True)
import joblib
scaler_path = os.path.join(MODEL_PATH, 'scaler.joblib')
joblib.dump(scaler, scaler_path)
print(f"\n[INFO] Scaler saved to {scaler_path}")

logger.log_param("scaler_type", "RobustScaler")
logger.log_param("n_features", X_train_scaled.shape[1])
logger.log_param("scale_pos_weight", scale_pos_weight)

mem_monitor.log(force=True)


FEATURE SCALING & CLASS WEIGHTS
[INFO] Checking for data leakage columns...
  [DROP] Removing leakage column: 'category'
  [DROP] Removing leakage column: 'subcategory '
[INFO] Processing non-numeric columns...
[INFO] Numeric columns: 31
[INFO] Non-numeric columns: ['flgs', 'proto', 'saddr', 'sport', 'daddr', 'dport', 'state', 'smac', 'dmac', 'soui', 'doui', 'sco', 'dco']
  [ENCODE] flgs: categorical → label encoded
  [ENCODE] proto: categorical → label encoded
  [ENCODE] saddr: categorical → label encoded
  [CONVERT] sport: string port → numeric
  [DROP] daddr: high cardinality string
  [CONVERT] dport: string port → numeric
  [ENCODE] state: categorical → label encoded
  [ENCODE] smac: categorical → label encoded
  [ENCODE] dmac: categorical → label encoded
  [ENCODE] soui: categorical → label encoded
  [ENCODE] doui: categorical → label encoded
  [ENCODE] sco: categorical → label encoded
  [ENCODE] dco: categorical → label encoded

[INFO] Checking for all-NaN / mostly-NaN columns..

{'total_gb': 31.350616455078125,
 'available_gb': 30.02408218383789,
 'used_gb': 2.576282501220703,
 'process_rss_gb': 2.3055458068847656,
 'process_vms_gb': 7.229122161865234,
 'percent': 4.2}

In [11]:
# ==============================================================================
# SECTION 8 — FEATURE SCALING & CLASS WEIGHTS
# ==============================================================================

from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

print("\n" + "="*60)
print("FEATURE SCALING & CLASS WEIGHTS")
print("="*60)

# STEP 1: Handle non-numeric columns properly
print("[INFO] Processing non-numeric columns...")

# Identify column types
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
non_numeric_cols = [c for c in X_train.columns if c not in numeric_cols]

print(f"[INFO] Numeric columns: {len(numeric_cols)}")
print(f"[INFO] Non-numeric columns: {non_numeric_cols}")

# Process each non-numeric column
for col in non_numeric_cols:
    # Check if it's a timestamp
    if 'time' in col.lower() or pd.api.types.is_datetime64_any_dtype(X_train[col]):
        print(f"  [CONVERT] {col}: datetime → unix timestamp")
        for df in [X_train, X_val, X_test]:
            df[col] = pd.to_numeric(pd.to_datetime(df[col], errors='coerce')) / 10**9

    # Check if it's a port number stored as string
    elif col in ['sport', 'dport']:
        print(f"  [CONVERT] {col}: string port → numeric")
        for df in [X_train, X_val, X_test]:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # Check if it's categorical (proto, state, flgs, etc.)
    elif X_train[col].nunique() < 50:
        print(f"  [ENCODE] {col}: categorical → label encoded")
        le = LabelEncoder()
        all_values = pd.concat([X_train[col], X_val[col], X_test[col]]).astype(str)
        le.fit(all_values)
        X_train[col] = le.transform(X_train[col].astype(str))
        X_val[col] = le.transform(X_val[col].astype(str))
        X_test[col] = le.transform(X_test[col].astype(str))

    # High-cardinality strings — drop them
    else:
        print(f"  [DROP] {col}: high cardinality string")
        X_train = X_train.drop(columns=[col])
        X_val = X_val.drop(columns=[col])
        X_test = X_test.drop(columns=[col])

# STEP 2: Drop columns that are ALL NaN or mostly NaN (>90%)
print("\n[INFO] Checking for all-NaN / mostly-NaN columns...")

# Collect columns to drop from ALL dataframes
cols_to_drop = set()

for df_name, df in [("train", X_train), ("val", X_val), ("test", X_test)]:
    nan_ratio = df.isna().mean()
    all_nan_cols = nan_ratio[nan_ratio == 1.0].index.tolist()
    mostly_nan_cols = nan_ratio[nan_ratio > 0.9].index.tolist()

    cols_to_drop.update(all_nan_cols)
    cols_to_drop.update(mostly_nan_cols)

    if all_nan_cols:
        print(f"  [{df_name}] Found ALL-NaN columns: {all_nan_cols}")
    if mostly_nan_cols:
        print(f"  [{df_name}] Found mostly-NaN columns: {mostly_nan_cols}")

# Drop collected columns from ALL dataframes at once (avoid KeyError)
if cols_to_drop:
    cols_to_drop = list(cols_to_drop)
    print(f"\n[INFO] Dropping columns from all sets: {cols_to_drop}")

    # Only drop columns that actually exist in each dataframe
    for df_name, df in [("train", X_train), ("val", X_val), ("test", X_test)]:
        existing_cols = [c for c in cols_to_drop if c in df.columns]
        if existing_cols:
            df.drop(columns=existing_cols, inplace=True)
            print(f"  [{df_name}] Dropped: {existing_cols}")

# STEP 3: Align columns across train/val/test (in case some were dropped)
common_cols = list(set(X_train.columns) & set(X_val.columns) & set(X_test.columns))
X_train = X_train[common_cols]
X_val = X_val[common_cols]
X_test = X_test[common_cols]

print(f"\n[INFO] Common columns after cleaning: {len(common_cols)}")

# STEP 4: Handle remaining NaN/Inf values
print("\n[INFO] Handling remaining NaN/Inf values...")

for df_name, df in [("train", X_train), ("val", X_val), ("test", X_test)]:
    nan_count = df.isna().sum().sum()

    if nan_count > 0:
        print(f"  [{df_name}] Filling {nan_count:,} NaN values with median")
        median_vals = df.median(numeric_only=True)
        df.fillna(median_vals, inplace=True)

    # Handle Inf values
    numeric_df = df.select_dtypes(include=[np.number])
    inf_mask = np.isinf(numeric_df)
    inf_count = inf_mask.sum().sum()

    if inf_count > 0:
        print(f"  [{df_name}] Replacing {inf_count:,} Inf values")
        for col in numeric_df.columns:
            col_inf = inf_mask[col]
            if col_inf.any():
                max_finite = numeric_df.loc[~col_inf, col].max()
                df.loc[col_inf, col] = max_finite

# STEP 5: Verify all columns are numeric
print("\n[INFO] Verifying all columns are numeric...")
remaining_non_numeric = X_train.select_dtypes(exclude=[np.number]).columns.tolist()
if remaining_non_numeric:
    print(f"[WARN] Still non-numeric columns: {remaining_non_numeric} — dropping them")
    X_train = X_train.drop(columns=remaining_non_numeric)
    X_val = X_val.drop(columns=remaining_non_numeric)
    X_test = X_test.drop(columns=remaining_non_numeric)
else:
    print("[INFO] All columns are now numeric ✓")

print(f"[INFO] Final shapes: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")

# STEP 6: Scale features
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"\n[INFO] Features scaled: {X_train_scaled.shape[1]} features")

# STEP 7: Compute class weights — CORRECTED for XGBoost
print("\n[INFO] Computing class weights...")

classes = np.unique(y_train)
class_counts = np.bincount(y_train.astype(int))

print(f"[INFO] Class distribution: {dict(zip(classes, class_counts))}")

# Sklearn-style class weights
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = {int(c): w for c, w in zip(classes, weights)}

print(f"[INFO] Sklearn class weights:")
for cls, weight in class_weight_dict.items():
    print(f"  Class {cls}: {weight:.4f}")

# XGBoost scale_pos_weight — CORRECTED
n_negative = (y_train == 0).sum()  # Benign
n_positive = (y_train == 1).sum()  # Attack

scale_pos_weight = n_positive / n_negative if n_negative > 0 else 1.0

print(f"\n[INFO] XGBoost scale_pos_weight:")
print(f"  Negative class (benign=0): {n_negative:,}")
print(f"  Positive class (attack=1): {n_positive:,}")
print(f"  scale_pos_weight = {n_positive}/{n_negative} = {scale_pos_weight:.2f}")

# STEP 8: Save scaler
# FIX: Ensure model directory exists before saving
os.makedirs(MODEL_PATH, exist_ok=True)

import joblib
scaler_path = os.path.join(MODEL_PATH, 'scaler.joblib')
joblib.dump(scaler, scaler_path)
print(f"\n[INFO] Scaler saved to {scaler_path}")

logger.log_param("scaler_type", "RobustScaler")
logger.log_param("n_features", X_train_scaled.shape[1])
logger.log_param("scale_pos_weight", scale_pos_weight)

mem_monitor.log(force=True)


FEATURE SCALING & CLASS WEIGHTS
[INFO] Processing non-numeric columns...
[INFO] Numeric columns: 43
[INFO] Non-numeric columns: []

[INFO] Checking for all-NaN / mostly-NaN columns...

[INFO] Common columns after cleaning: 43

[INFO] Handling remaining NaN/Inf values...

[INFO] Verifying all columns are numeric...
[INFO] All columns are now numeric ✓
[INFO] Final shapes: Train=(9442, 43), Val=(4720, 43), Test=(4720, 43)

[INFO] Features scaled: 43 features

[INFO] Computing class weights...
[INFO] Class distribution: {np.uint8(0): np.int64(4721), np.uint8(1): np.int64(4721)}
[INFO] Sklearn class weights:
  Class 0: 1.0000
  Class 1: 1.0000

[INFO] XGBoost scale_pos_weight:
  Negative class (benign=0): 4,721
  Positive class (attack=1): 4,721
  scale_pos_weight = 4721/4721 = 1.00

[INFO] Scaler saved to /kaggle/working/models/scaler.joblib
2026-05-11 20:14:03,812 [INFO] [PARAM] scaler_type = RobustScaler
2026-05-11 20:14:03,813 [INFO] [PARAM] n_features = 43
2026-05-11 20:14:03,813 [IN

{'total_gb': 31.350616455078125,
 'available_gb': 30.025074005126953,
 'used_gb': 2.5753211975097656,
 'process_rss_gb': 2.3055496215820312,
 'process_vms_gb': 7.229122161865234,
 'percent': 4.2}

In [12]:
# ==============================================================================
# SECTION 9 — MODEL TRAINING WITH VERIFICATION (FINAL v2)
# ==============================================================================

import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
import optuna

print("\n" + "="*60)
print("MODEL TRAINING")
print("="*60)

# Configuration
MODEL_TYPE = 'xgboost'
USE_OPTUNA = True
N_OPTUNA_TRIALS = 10

base_params = {
    'xgboost': {
        'scale_pos_weight': scale_pos_weight,
        'max_depth': 6,
        'learning_rate': 0.1,
        'n_estimators': 100,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'random_state': RANDOM_SEED,
        'n_jobs': -1,
        'eval_metric': 'logloss',
        'use_label_encoder': False
    },
    'random_forest': {
        'class_weight': 'balanced',
        'n_estimators': 100,
        'max_depth': 20,
        'random_state': RANDOM_SEED,
        'n_jobs': -1
    },
    'logistic': {
        'class_weight': 'balanced',
        'max_iter': 1000,
        'random_state': RANDOM_SEED,
        'n_jobs': -1
    }
}

def objective(trial):
    """Optuna objective for hyperparameter tuning."""
    if MODEL_TYPE == 'xgboost':
        params = {
            'scale_pos_weight': scale_pos_weight,
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'n_estimators': trial.suggest_int('n_estimators', 50, 300),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'random_state': RANDOM_SEED,
            'n_jobs': -1,
            'eval_metric': 'logloss',
            'use_label_encoder': False
        }
        model = xgb.XGBClassifier(**params)

    model.fit(X_train_scaled, y_train)
    y_val_pred = model.predict(X_val_scaled)
    return f1_score(y_val, y_val_pred)

# Check if model exists and is valid
model_path = os.path.join(MODEL_PATH, f'{MODEL_TYPE}_model.joblib')
model_exists = os.path.exists(model_path)
model_valid = False

if model_exists:
    try:
        test_model = joblib.load(model_path)
        # Verify model works
        test_pred = test_model.predict(X_val_scaled[:5])
        test_acc = accuracy_score(y_val[:5], test_pred)
        if test_acc > 0.8:  # Sanity check
            model_valid = True
            print("[INFO] Existing model verified and valid")
        else:
            print("[WARN] Existing model gives suspicious accuracy, retraining...")
    except Exception as e:
        print(f"[WARN] Failed to load existing model: {e}")

# Train if needed
if not model_valid:
    print(f"[INFO] Training new {MODEL_TYPE} model...")

    if USE_OPTUNA:
        print(f"[INFO] Optuna tuning ({N_OPTUNA_TRIALS} trials)...")
        study = optuna.create_study(direction='maximize')
        study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

        print(f"[OPTUNA] Best F1: {study.best_value:.4f}")
        best_params = base_params[MODEL_TYPE].copy()
        best_params.update(study.best_params)

        if MODEL_TYPE == 'xgboost':
            model = xgb.XGBClassifier(**best_params)
    else:
        if MODEL_TYPE == 'xgboost':
            model = xgb.XGBClassifier(**base_params['xgboost'])

    # Train
    start_time = time.time()
    model.fit(X_train_scaled, y_train)
    training_time = time.time() - start_time

    print(f"[INFO] Training complete in {training_time:.1f}s")

    # Verify training worked
    train_pred = model.predict(X_train_scaled)
    train_acc = accuracy_score(y_train, train_pred)
    print(f"[INFO] Train accuracy: {train_acc:.4f}")

    if train_acc < 0.9:
        raise ValueError(f"Training failed! Accuracy {train_acc:.4f} is too low")

    # Save with verification
    os.makedirs(MODEL_PATH, exist_ok=True)
    joblib.dump(model, model_path)

    # Verify save
    loaded = joblib.load(model_path)
    verify_pred = loaded.predict(X_val_scaled[:10])
    verify_acc = accuracy_score(y_val[:10], verify_pred)
    print(f"[INFO] Save verified: {verify_acc:.4f} accuracy on sample")

    logger.log_param("model_type", MODEL_TYPE)
    logger.log_param("training_time_sec", training_time)
    logger.log_param("train_accuracy", train_acc)

else:
    model = joblib.load(model_path)
    print(f"[INFO] Loaded valid model from {model_path}")

mem_monitor.log(force=True)


MODEL TRAINING
[INFO] Training new xgboost model...
[INFO] Optuna tuning (10 trials)...


  0%|          | 0/10 [00:00<?, ?it/s]

[OPTUNA] Best F1: 0.9992
[INFO] Training complete in 0.2s
[INFO] Train accuracy: 0.9998
[INFO] Save verified: 1.0000 accuracy on sample
2026-05-11 20:14:07,320 [INFO] [PARAM] model_type = xgboost
2026-05-11 20:14:07,321 [INFO] [PARAM] training_time_sec = 0.16604042053222656
2026-05-11 20:14:07,322 [INFO] [PARAM] train_accuracy = 0.9997881804702393
2026-05-11 20:14:07,322 [INFO] [EVENT] memory_status | {"ram_used_gb": 2.57, "ram_available_gb": 30.03, "process_rss_gb": 2.34, "peak_ram_gb": 2.37, "ram_percent": 4.2}


{'total_gb': 31.350616455078125,
 'available_gb': 30.026233673095703,
 'used_gb': 2.5742034912109375,
 'process_rss_gb': 2.336071014404297,
 'process_vms_gb': 7.418743133544922,
 'percent': 4.2}

In [13]:
# ==============================================================================
# SECTION 10 — EVALUATION WITH BALANCED METRICS
# ==============================================================================

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             classification_report, roc_curve, precision_recall_curve)
import matplotlib.pyplot as plt

print("\n" + "="*60)
print("MODEL EVALUATION")
print("="*60)

# Get predictions
y_train_proba = model.predict_proba(X_train_scaled)[:, 1]
y_val_proba = model.predict_proba(X_val_scaled)[:, 1]
y_test_proba = model.predict_proba(X_test_scaled)[:, 1]

def find_optimal_threshold(y_true, y_proba, metric='f1'):
    """Find optimal threshold using balanced validation set."""
    thresholds = np.arange(0.01, 1.0, 0.01)
    scores = []

    for thresh in thresholds:
        y_pred = (y_proba >= thresh).astype(int)
        if metric == 'f1':
            score = f1_score(y_true, y_pred)
        elif metric == 'recall':
            score = recall_score(y_true, y_pred)
        elif metric == 'youden':
            tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
            sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
            score = sensitivity + specificity - 1
        scores.append(score)

    best_idx = np.argmax(scores)
    return thresholds[best_idx], scores[best_idx]

def evaluate_at_threshold(y_true, y_proba, threshold, set_name):
    """Evaluate model at given threshold."""
    y_pred = (y_proba >= threshold).astype(int)

    # Handle edge case where all predictions are same class
    if len(np.unique(y_pred)) < 2:
        print(f"\n[WARNING] {set_name}: All predictions are class {np.unique(y_pred)[0]}!")
        print(f"  This indicates threshold {threshold} is too extreme.")

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_true, y_proba)
    pr_auc = average_precision_score(y_true, y_proba)

    cm = confusion_matrix(y_true, y_pred)

    print(f"\n{'='*60}")
    print(f"EVALUATION ON {set_name.upper()} SET")
    print(f"{'='*60}")
    print(f"[METRICS] Threshold: {threshold:.3f}")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1 Score:  {f1:.4f}")
    print(f"  ROC-AUC:   {roc_auc:.4f}")
    print(f"  PR-AUC:    {pr_auc:.4f}")

    print(f"\n[CONFUSION MATRIX]")
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        print(f"  TN: {tn:>6} | FP: {fp:>6}")
        print(f"  FN: {fn:>6} | TP: {tp:>6}")
        print(f"  False Positive Rate: {fp/(fp+tn):.4f}" if (fp+tn) > 0 else "  FPR: N/A")
        print(f"  False Negative Rate: {fn/(fn+tp):.4f}" if (fn+tp) > 0 else "  FNR: N/A")
    else:
        print(cm)

    print(f"\n[CLASSIFICATION REPORT]")
    print(classification_report(y_true, y_pred, target_names=['Benign', 'Attack'], zero_division=0))

    return {
        'threshold': threshold,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'roc_auc': roc_auc,
        'pr_auc': pr_auc,
        'confusion_matrix': cm
    }

# Find optimal thresholds on BALANCED validation set
print("\n[THRESHOLD OPTIMIZATION]")
print("Using BALANCED validation set for threshold tuning...")

f1_threshold, f1_score_val = find_optimal_threshold(y_val, y_val_proba, metric='f1')
recall_threshold, recall_score_val = find_optimal_threshold(y_val, y_val_proba, metric='recall')
youden_threshold, youden_score_val = find_optimal_threshold(y_val, y_val_proba, metric='youden')

print(f"  F1-optimal threshold: {f1_threshold:.2f} (F1={f1_score_val:.4f})")
print(f"  Recall-prioritized threshold: {recall_threshold:.2f}")
print(f"  Youden J threshold: {youden_threshold:.2f} (J={youden_score_val:.4f})")

# Use F1-optimal threshold for evaluation
optimal_threshold = f1_threshold

# Evaluate on all sets
val_metrics = evaluate_at_threshold(y_val, y_val_proba, optimal_threshold, "validation (balanced)")
test_metrics = evaluate_at_threshold(y_test, y_test_proba, optimal_threshold, "test (balanced)")

# Also evaluate at 0.5 for comparison
print("\n" + "="*60)
print("COMPARISON: THRESHOLD 0.5 (DEFAULT)")
print("="*60)
val_metrics_05 = evaluate_at_threshold(y_val, y_val_proba, 0.5, "validation at 0.5")
test_metrics_05 = evaluate_at_threshold(y_test, y_test_proba, 0.5, "test at 0.5")

# Plot ROC and PR curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
for y_true, y_proba, name in [(y_val, y_val_proba, 'Val'), (y_test, y_test_proba, 'Test')]:
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    auc = roc_auc_score(y_true, y_proba)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', linewidth=2)

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# PR Curve
for y_true, y_proba, name in [(y_val, y_val_proba, 'Val'), (y_test, y_test_proba, 'Test')]:
    precision, recall, _ = precision_recall_curve(y_true, y_proba)
    auc = average_precision_score(y_true, y_proba)
    axes[1].plot(recall, precision, label=f'{name} (AP={auc:.3f})', linewidth=2)

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(WORKING_PATH, 'roc_pr_curves.png'), dpi=150, bbox_inches='tight')
plt.close()

# Feature importance (if available)
if hasattr(model, 'feature_importances_') and hasattr(fe, 'feature_list'):
    importances = model.feature_importances_
    feature_names = fe.feature_list

    if len(importances) == len(feature_names):
        top_idx = np.argsort(importances)[::-1][:10]
        print(f"\n[TOP 10 FEATURES]")
        for i, idx in enumerate(top_idx, 1):
            print(f"  {i:2d}. {feature_names[idx]:20s}: {importances[idx]:.4f}")

        # Plot
        plt.figure(figsize=(10, 6))
        plt.barh(range(10), importances[top_idx][::-1])
        plt.yticks(range(10), [feature_names[i] for i in top_idx[::-1]])
        plt.xlabel('Importance')
        plt.title('Top 10 Feature Importances')
        plt.tight_layout()
        plt.savefig(os.path.join(WORKING_PATH, 'feature_importance.png'), dpi=150, bbox_inches='tight')
        plt.close()

# Log final metrics
logger.log_param("optimal_threshold", optimal_threshold)
logger.log_param("val_f1", val_metrics['f1'])
logger.log_param("val_precision", val_metrics['precision'])
logger.log_param("val_recall", val_metrics['recall'])
logger.log_param("test_f1", test_metrics['f1'])
logger.log_param("test_precision", test_metrics['precision'])
logger.log_param("test_recall", test_metrics['recall'])

print("\n" + "="*60)
print("EVALUATION COMPLETE")
print("="*60)
print(f"Optimal threshold: {optimal_threshold:.3f}")
print(f"Validation F1: {val_metrics['f1']:.4f}")
print(f"Test F1: {test_metrics['f1']:.4f}")

mem_monitor.log(force=True)


MODEL EVALUATION

[THRESHOLD OPTIMIZATION]
Using BALANCED validation set for threshold tuning...
  F1-optimal threshold: 0.48 (F1=0.9992)
  Recall-prioritized threshold: 0.01
  Youden J threshold: 0.48 (J=0.9983)

EVALUATION ON VALIDATION (BALANCED) SET
[METRICS] Threshold: 0.480
  Accuracy:  0.9992
  Precision: 0.9996
  Recall:    0.9987
  F1 Score:  0.9992
  ROC-AUC:   0.9999
  PR-AUC:    0.9999

[CONFUSION MATRIX]
  TN:   2359 | FP:      1
  FN:      3 | TP:   2357
  False Positive Rate: 0.0004
  False Negative Rate: 0.0013

[CLASSIFICATION REPORT]
              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00      2360
      Attack       1.00      1.00      1.00      2360

    accuracy                           1.00      4720
   macro avg       1.00      1.00      1.00      4720
weighted avg       1.00      1.00      1.00      4720


EVALUATION ON TEST (BALANCED) SET
[METRICS] Threshold: 0.480
  Accuracy:  0.9989
  Precision: 0.9996
  Recall:   

{'total_gb': 31.350616455078125,
 'available_gb': 30.02715301513672,
 'used_gb': 2.5732345581054688,
 'process_rss_gb': 2.3365249633789062,
 'process_vms_gb': 7.418743133544922,
 'percent': 4.2}

In [14]:
TORCH_AVAILABLE = True
USE_AUTOENCODER = True  # If you want to actually train it

In [15]:
# ==============================================================================
# SECTION 11 — OPTIONAL DEEP AUTOENCODER
# ==============================================================================

# Import torch modules at module level before any class definitions
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

class IoTAutoencoder:
    """
    GPU-accelerated autoencoder for anomaly detection.
    Trains ONLY on benign traffic. Dynamic batch size and precision based on VRAM.
    """

    def __init__(self, input_dim: int, config: Dict = None):
        self.config = config or AUTOENCODER_CONFIG
        self.input_dim = input_dim
        self.device = self._detect_device()
        self.model = None
        self.scaler = StandardScaler()
        self.history = {'train_loss': [], 'val_loss': []}

        # Dynamic batch size based on VRAM
        if self.device.type == 'cuda':
            vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
            if vram_gb < 4:
                self.config['batch_size'] = 256
            elif vram_gb < 8:
                self.config['batch_size'] = 512
            else:
                self.config['batch_size'] = 1024
            print(f"[AUTOENCODER] VRAM: {vram_gb:.1f}GB, Batch size: {self.config['batch_size']}")
        else:
            self.config['batch_size'] = 512

        # Mixed precision
        self.use_amp = self.device.type == 'cuda' and hasattr(torch.cuda, 'amp')
        self.scaler_amp = torch.cuda.amp.GradScaler() if self.use_amp else None

        print(f"[AUTOENCODER] Device: {self.device}, AMP: {self.use_amp}")

    def _detect_device(self):
        """Auto-detect CUDA and select appropriate device."""
        if torch.cuda.is_available():
            return torch.device('cuda')
        return torch.device('cpu')

    class AutoencoderModel(nn.Module):
        """PyTorch autoencoder architecture."""

        def __init__(self, input_dim: int, hidden_dims: List[int], dropout: float = 0.2):
            super().__init__()

            # Encoder
            encoder_layers = []
            prev_dim = input_dim
            for dim in hidden_dims[:len(hidden_dims)//2 + 1]:
                encoder_layers.extend([
                    nn.Linear(prev_dim, dim),
                    nn.ReLU(),
                    nn.Dropout(dropout),
                    nn.BatchNorm1d(dim)
                ])
                prev_dim = dim
            self.encoder = nn.Sequential(*encoder_layers)

            # Decoder
            decoder_layers = []
            for dim in hidden_dims[len(hidden_dims)//2 + 1:]:
                decoder_layers.extend([
                    nn.Linear(prev_dim, dim),
                    nn.ReLU(),
                    nn.Dropout(dropout),
                    nn.BatchNorm1d(dim)
                ])
                prev_dim = dim
            decoder_layers.append(nn.Linear(prev_dim, input_dim))
            self.decoder = nn.Sequential(*decoder_layers)

        def forward(self, x):
            encoded = self.encoder(x)
            decoded = self.decoder(encoded)
            return decoded

    class IoTDataset(Dataset):
        """Dataset for benign traffic only."""

        def __init__(self, X: np.ndarray):
            self.X = torch.FloatTensor(X)

        def __len__(self):
            return len(self.X)

        def __getitem__(self, idx):
            return self.X[idx]

    def fit(self, X_train: np.ndarray, X_val: Optional[np.ndarray] = None) -> Dict:
        """
        Train autoencoder on benign traffic only.
        """
        print("\n" + "="*60)
        print("TRAINING DEEP AUTOENCODER")
        print("="*60)

        # Scale data
        X_train_scaled = self.scaler.fit_transform(X_train)
        if X_val is not None:
            X_val_scaled = self.scaler.transform(X_val)

        # Create dataloaders
        train_dataset = self.IoTDataset(X_train_scaled)
        train_loader = DataLoader(train_dataset, batch_size=self.config['batch_size'], 
                                 shuffle=True, num_workers=0)

        if X_val is not None:
            val_dataset = self.IoTDataset(X_val_scaled)
            val_loader = DataLoader(val_dataset, batch_size=self.config['batch_size'], 
                                   shuffle=False, num_workers=0)

        # Initialize model
        self.model = self.AutoencoderModel(
            self.input_dim, 
            self.config['hidden_dims'],
            self.config['dropout']
        ).to(self.device)

        optimizer = optim.Adam(self.model.parameters(), lr=self.config['learning_rate'])
        criterion = nn.MSELoss()

        # Training loop with early stopping
        best_val_loss = float('inf')
        patience_counter = 0

        start_time = time.time()

        for epoch in range(self.config['max_epochs']):
            # Training
            self.model.train()
            train_losses = []

            for batch in train_loader:
                batch = batch.to(self.device)

                optimizer.zero_grad()

                if self.use_amp:
                    with torch.cuda.amp.autocast():
                        outputs = self.model(batch)
                        loss = criterion(outputs, batch)
                    self.scaler_amp.scale(loss).backward()
                    self.scaler_amp.step(optimizer)
                    self.scaler_amp.update()
                else:
                    outputs = self.model(batch)
                    loss = criterion(outputs, batch)
                    loss.backward()
                    optimizer.step()

                train_losses.append(loss.item())

            avg_train_loss = np.mean(train_losses)
            self.history['train_loss'].append(avg_train_loss)

            # Validation
            if X_val is not None:
                self.model.eval()
                val_losses = []

                with torch.no_grad():
                    for batch in val_loader:
                        batch = batch.to(self.device)

                        if self.use_amp:
                            with torch.cuda.amp.autocast():
                                outputs = self.model(batch)
                                loss = criterion(outputs, batch)
                        else:
                            outputs = self.model(batch)
                            loss = criterion(outputs, batch)

                        val_losses.append(loss.item())

                avg_val_loss = np.mean(val_losses)
                self.history['val_loss'].append(avg_val_loss)

                # Early stopping
                if avg_val_loss < best_val_loss:
                    best_val_loss = avg_val_loss
                    patience_counter = 0
                    # Save best model
                    torch.save(self.model.state_dict(), os.path.join(MODEL_PATH, 'autoencoder_best.pt'))
                else:
                    patience_counter += 1

                if patience_counter >= self.config['patience']:
                    print(f"  Early stopping at epoch {epoch + 1}")
                    break

                if (epoch + 1) % 10 == 0:
                    print(f"  Epoch {epoch + 1}: Train Loss = {avg_train_loss:.6f}, Val Loss = {avg_val_loss:.6f}")
            else:
                if (epoch + 1) % 10 == 0:
                    print(f"  Epoch {epoch + 1}: Train Loss = {avg_train_loss:.6f}")

        training_time = time.time() - start_time
        print(f"\n[AUTOENCODER] Training complete in {training_time:.1f}s")

        # Plot training curves
        plt.figure(figsize=(10, 5))
        plt.plot(self.history['train_loss'], label='Train Loss', linewidth=2)
        if self.history['val_loss']:
            plt.plot(self.history['val_loss'], label='Val Loss', linewidth=2)
        plt.xlabel('Epoch')
        plt.ylabel('MSE Loss')
        plt.title('Autoencoder Training Curves')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(WORKING_PATH, 'autoencoder_training.png'), dpi=150, bbox_inches='tight')
        plt.close()

        logger.log_event("autoencoder_trained", {
            'epochs': len(self.history['train_loss']),
            'final_train_loss': avg_train_loss,
            'training_time_sec': training_time
        })

        return self.history

    def get_anomaly_scores(self, X: np.ndarray) -> np.ndarray:
        """
        Get reconstruction error as anomaly scores.
        Higher scores = more anomalous.
        """
        if self.model is None:
            raise ValueError("Model not trained")

        X_scaled = self.scaler.transform(X)
        dataset = self.IoTDataset(X_scaled)
        loader = DataLoader(dataset, batch_size=self.config['batch_size'], shuffle=False)

        self.model.eval()
        scores = []

        with torch.no_grad():
            for batch in loader:
                batch = batch.to(self.device)

                if self.use_amp:
                    with torch.cuda.amp.autocast():
                        outputs = self.model(batch)
                        loss = nn.MSELoss(reduction='none')(outputs, batch)
                else:
                    outputs = self.model(batch)
                    loss = nn.MSELoss(reduction='none')(outputs, batch)

                # Per-sample reconstruction error
                sample_loss = loss.mean(dim=1).cpu().numpy()
                scores.extend(sample_loss)

        return np.array(scores)

    def plot_anomaly_distribution(self, X_benign: np.ndarray, X_attack: np.ndarray) -> None:
        """Plot anomaly score distributions for benign vs attack traffic."""
        scores_benign = self.get_anomaly_scores(X_benign)
        scores_attack = self.get_anomaly_scores(X_attack)

        plt.figure(figsize=(12, 5))

        plt.subplot(1, 2, 1)
        plt.hist(scores_benign, bins=50, alpha=0.7, label='Benign', color='green', density=True)
        plt.hist(scores_attack, bins=50, alpha=0.7, label='Attack', color='red', density=True)
        plt.xlabel('Reconstruction Error')
        plt.ylabel('Density')
        plt.title('Anomaly Score Distribution')
        plt.legend()
        plt.yscale('log')

        plt.subplot(1, 2, 2)
        plt.boxplot([scores_benign, scores_attack], labels=['Benign', 'Attack'])
        plt.ylabel('Reconstruction Error')
        plt.title('Anomaly Score Boxplot')
        plt.yscale('log')

        plt.tight_layout()
        plt.savefig(os.path.join(WORKING_PATH, 'anomaly_distribution.png'), dpi=150, bbox_inches='tight')
        plt.close()

        # Plotly version
        fig = go.Figure()
        fig.add_trace(go.Histogram(x=scores_benign, name='Benign', opacity=0.7, 
                                  marker_color='green', nbinsx=50))
        fig.add_trace(go.Histogram(x=scores_attack, name='Attack', opacity=0.7, 
                                  marker_color='red', nbinsx=50))
        fig.update_layout(
            title='Anomaly Score Distribution',
            xaxis_title='Reconstruction Error',
            yaxis_title='Count',
            barmode='overlay',
            template='plotly_white',
            height=500
        )
        fig.write_html(os.path.join(WORKING_PATH, 'anomaly_distribution_interactive.html'))

        print(f"  Benign mean score: {scores_benign.mean():.4f}")
        print(f"  Attack mean score: {scores_attack.mean():.4f}")
        print(f"  Separation ratio: {scores_attack.mean() / (scores_benign.mean() + 1e-8):.2f}x")

# Train autoencoder if enabled
if USE_AUTOENCODER:
    os.makedirs(MODEL_PATH, exist_ok=True)

    # Train only on benign samples
    benign_mask = y_train == 0
    X_train_benign = X_train[benign_mask]

    autoencoder = IoTAutoencoder(input_dim=X_train.shape[1])
    autoencoder.fit(X_train_benign, X_val)

    # Evaluate anomaly detection
    benign_test = X_test[y_test == 0]
    attack_test = X_test[y_test == 1]

    if len(benign_test) > 0 and len(attack_test) > 0:
        autoencoder.plot_anomaly_distribution(benign_test, attack_test)

    mem_monitor.log(force=True)

[AUTOENCODER] Device: cpu, AMP: False

TRAINING DEEP AUTOENCODER
  Epoch 10: Train Loss = 0.345378, Val Loss = 18.884295
  Epoch 20: Train Loss = 0.269089, Val Loss = 14.055279
  Epoch 30: Train Loss = 0.241820, Val Loss = 12.547409
  Epoch 40: Train Loss = 0.218809, Val Loss = 11.530381
  Epoch 50: Train Loss = 0.203739, Val Loss = 12.762084

[AUTOENCODER] Training complete in 5.0s
2026-05-11 20:14:23,994 [INFO] [EVENT] autoencoder_trained | {"epochs": 50, "final_train_loss": 0.20373921543359758, "training_time_sec": 4.952154159545898}


/tmp/ipykernel_16/2440440153.py:282: MatplotlibDeprecationWarning:

The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.



  Benign mean score: 0.1091
  Attack mean score: 55.6978
  Separation ratio: 510.63x
2026-05-11 20:14:25,079 [INFO] [EVENT] memory_status | {"ram_used_gb": 2.79, "ram_available_gb": 29.81, "process_rss_gb": 2.62, "peak_ram_gb": 2.62, "ram_percent": 4.9}


In [16]:
# ==============================================================================
# SECTION 12 — CROSS-DATASET VALIDATION & DEPLOYMENT PREP (FIXED v3)
# ==============================================================================

from datetime import datetime
import json

class CrossDatasetPipeline:
    """
    Validate model on held-out datasets and prepare deployment artifacts.
    """

    def __init__(self, feature_engineer, model, scaler=None):
        self.fe = feature_engineer
        self.model = model
        self.scaler = scaler
        self.validation_results = []

    def validate_on_dataset(self, X_new, y_new, dataset_name='held_out', skip_fe=False, skip_scale=False):
        """
        Validate on a completely new dataset.

        Parameters:
        -----------
        X_new : pd.DataFrame or np.ndarray
            Input features (raw or pre-scaled)
        y_new : array-like
            True labels
        dataset_name : str
            Name for logging
        skip_fe : bool
            If True, skip feature engineering (X_new is already processed)
        skip_scale : bool
            If True, skip scaling (X_new is already scaled)
        """
        print(f"\n[VALIDATION] Testing on {dataset_name}...")

        X_processed = X_new

        # Step 1: Feature Engineering (only if not skipped)
        if not skip_fe and self.fe is not None:
            if hasattr(self.fe, 'transform'):
                X_processed = self.fe.transform(X_processed)
            elif hasattr(self.fe, 'fit_transform'):
                print("  [WARN] Using fit_transform instead of transform")
                X_processed = self.fe.fit_transform(X_processed)
            else:
                print("  [WARN] No transform method, using X_new as-is")

        # Step 2: Scaling (only if not skipped and scaler provided)
        if not skip_scale and self.scaler is not None and hasattr(self.scaler, 'transform'):
            X_processed = self.scaler.transform(X_processed)

        # Step 3: Predict
        y_proba = self.model.predict_proba(X_processed)[:, 1]
        y_pred = (y_proba >= 0.5).astype(int)

        from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

        acc = accuracy_score(y_new, y_pred)
        f1 = f1_score(y_new, y_pred)
        roc = roc_auc_score(y_new, y_proba)

        result = {
            'dataset': dataset_name,
            'accuracy': acc,
            'f1': f1,
            'roc_auc': roc,
            'n_samples': len(y_new)
        }

        self.validation_results.append(result)

        print(f"  Accuracy: {acc:.4f}")
        print(f"  F1: {f1:.4f}")
        print(f"  ROC-AUC: {roc:.4f}")

        return result

    def save_artifacts(self, path=None):
        """Save all artifacts for deployment."""
        if path is None:
            path = MODEL_PATH

        os.makedirs(path, exist_ok=True)

        artifacts = {
            'feature_engineer': {
                'feature_list': getattr(self.fe, 'feature_list', []),
                'n_features': len(getattr(self.fe, 'feature_list', [])),
            },
            'model_info': {
                'type': type(self.model).__name__,
                'model_params': self.model.get_params() if hasattr(self.model, 'get_params') else {},
                'best_iteration': getattr(self.model, 'best_iteration', None),
                'n_features_in': getattr(self.model, 'n_features_in_', None),
            },
            'validation_results': self.validation_results,
            'training_timestamp': datetime.now().isoformat(),
            'deployment_ready': len(self.validation_results) > 0
        }

        artifact_path = os.path.join(path, 'deployment_artifacts.json')
        with open(artifact_path, 'w') as f:
            json.dump(artifacts, f, indent=2, default=str)

        print(f"\n[ARTIFACTS] Saved to {artifact_path}")
        print(f"[ARTIFACTS] Model type: {artifacts['model_info']['type']}")
        print(f"[ARTIFACTS] Features: {artifacts['feature_engineer']['n_features']}")
        print(f"[ARTIFACTS] Validation datasets: {len(self.validation_results)}")

        return artifacts

# Save cross-dataset artifacts
cross_dataset = CrossDatasetPipeline(fe, model, scaler=scaler)

# FIX: X_test_scaled is already scaled, so skip both FE and scaling
cross_dataset.validate_on_dataset(
    X_test_scaled, y_test, 
    dataset_name='test_set',
    skip_fe=True,    # Already processed
    skip_scale=True  # Already scaled
)

# Save artifacts
cross_dataset.save_artifacts()

mem_monitor.log(force=True)


[VALIDATION] Testing on test_set...
  Accuracy: 0.9989
  F1: 0.9989
  ROC-AUC: 1.0000

[ARTIFACTS] Saved to /kaggle/working/models/deployment_artifacts.json
[ARTIFACTS] Model type: XGBClassifier
[ARTIFACTS] Features: 47
[ARTIFACTS] Validation datasets: 1
2026-05-11 20:14:25,152 [INFO] [EVENT] memory_status | {"ram_used_gb": 2.79, "ram_available_gb": 29.81, "process_rss_gb": 2.62, "peak_ram_gb": 2.62, "ram_percent": 4.9}


{'total_gb': 31.350616455078125,
 'available_gb': 29.81155014038086,
 'used_gb': 2.7888946533203125,
 'process_rss_gb': 2.6223411560058594,
 'process_vms_gb': 7.991130828857422,
 'percent': 4.9}

In [17]:
# ==============================================================================
# SECTION 13 — CONCEPT DRIFT ANALYSIS
# ==============================================================================

from scipy import stats
from scipy.spatial.distance import jensenshannon
import numpy as np

def analyze_concept_drift(train_data, val_data, test_data, feature_cols, threshold=0.05):
    """
    Analyze concept drift between train/val/test sets.
    Uses KS test and distribution distance metrics.
    """
    print("\n" + "="*60)
    print("CONCEPT DRIFT ANALYSIS")
    print("="*60)

    drift_results = {
        'ks_tests': {},
        'distribution_distances': {},
        'drift_detected': False,
        'drifted_features': []
    }

    for col in feature_cols:
        # Get numeric data only
        train_vals = pd.to_numeric(train_data[col], errors='coerce').dropna()
        val_vals = pd.to_numeric(val_data[col], errors='coerce').dropna()
        test_vals = pd.to_numeric(test_data[col], errors='coerce').dropna()

        if len(train_vals) == 0 or len(val_vals) == 0:
            continue

        # KS test: train vs val
        ks_stat, p_value = stats.ks_2samp(train_vals, val_vals)

        # Distribution distance (Jensen-Shannon)
        # Bin data and compute distance
        all_vals = np.concatenate([train_vals, val_vals])
        bins = np.linspace(all_vals.min(), all_vals.max(), 50)

        train_hist, _ = np.histogram(train_vals, bins=bins, density=True)
        val_hist, _ = np.histogram(val_vals, bins=bins, density=True)

        # Add small epsilon to avoid log(0)
        train_hist += 1e-10
        val_hist += 1e-10

        js_distance = jensenshannon(train_hist, val_hist)

        drift_results['ks_tests'][col] = {
            'statistic': ks_stat,
            'p_value': p_value,
            'significant': p_value < threshold
        }

        drift_results['distribution_distances'][col] = js_distance

        if p_value < threshold:
            drift_results['drift_detected'] = True
            drift_results['drifted_features'].append(col)

    # Summary
    n_drifted = len(drift_results['drifted_features'])
    n_total = len(feature_cols)

    print(f"[INFO] Features analyzed: {n_total}")
    print(f"[INFO] Features with significant drift (p < {threshold}): {n_drifted}")

    if n_drifted > 0:
        print(f"[WARN] Drift detected in: {drift_results['drifted_features'][:10]}")
        if n_drifted > 10:
            print(f"  ... and {n_drifted - 10} more")
    else:
        print("[INFO] No significant concept drift detected ✓")

    # Log results
    logger.log_param("drift_detected", drift_results['drift_detected'])
    logger.log_param("drifted_features_count", n_drifted)

    return drift_results

# Prepare data for drift analysis
# FIX: Create DataFrames from scaled data with column names
train_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
val_df = pd.DataFrame(X_val_scaled, columns=X_val.columns)
test_df = pd.DataFrame(X_test_scaled, columns=X_test.columns)

# Select feature columns (exclude non-numeric if any)
feature_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()

# Run drift analysis
drift_results = analyze_concept_drift(train_df, val_df, test_df, feature_cols)

# Plot drift for top features
if drift_results['drift_detected']:
    top_drift = sorted(drift_results['ks_tests'].items(), 
                       key=lambda x: x[1]['statistic'], reverse=True)[:5]

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    for i, (col, result) in enumerate(top_drift):
        if i < len(axes):
            axes[i].hist(train_df[col], bins=30, alpha=0.5, label='Train', density=True)
            axes[i].hist(val_df[col], bins=30, alpha=0.5, label='Val', density=True)
            axes[i].set_title(f'{col}\nKS={result["statistic"]:.3f}')
            axes[i].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(WORKING_PATH, 'concept_drift.png'), dpi=150, bbox_inches='tight')
    plt.close()

mem_monitor.log(force=True)


CONCEPT DRIFT ANALYSIS


/usr/local/lib/python3.12/dist-packages/numpy/lib/_histograms_impl.py:895: RuntimeWarning:

divide by zero encountered in divide

/usr/local/lib/python3.12/dist-packages/numpy/lib/_histograms_impl.py:895: RuntimeWarning:

invalid value encountered in divide



[INFO] Features analyzed: 43
[INFO] Features with significant drift (p < 0.05): 1
[WARN] Drift detected in: ['dur']
2026-05-11 20:14:25,557 [INFO] [PARAM] drift_detected = True
2026-05-11 20:14:25,558 [INFO] [PARAM] drifted_features_count = 1
2026-05-11 20:14:26,629 [INFO] [EVENT] memory_status | {"ram_used_gb": 2.8, "ram_available_gb": 29.8, "process_rss_gb": 2.63, "peak_ram_gb": 2.63, "ram_percent": 4.9}


{'total_gb': 31.350616455078125,
 'available_gb': 29.804481506347656,
 'used_gb': 2.7959365844726562,
 'process_rss_gb': 2.6265907287597656,
 'process_vms_gb': 7.996013641357422,
 'percent': 4.9}

In [18]:
# ==============================================================================
# SECTION 14 — MODEL ARTIFACTS SAVING (FIXED)
# ==============================================================================

from datetime import datetime
import json
import joblib

def save_model_artifacts(
    model,
    feature_cols: List[str],
    best_params: Dict,
    model_name: str = "xgboost_ids",
    output_dir: str = None
) -> Dict:
    """
    Save model artifacts for deployment.

    Parameters:
    -----------
    model : trained model object
        The trained model (XGBoost, LightGBM, etc.)
    feature_cols : List[str]
        List of feature column names
    best_params : Dict
        Best hyperparameters from tuning
    model_name : str
        Name for the saved model
    output_dir : str
        Directory to save artifacts
    """
    if output_dir is None:
        output_dir = MODEL_PATH

    os.makedirs(output_dir, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Save model
    model_path = os.path.join(output_dir, f'{model_name}_{timestamp}.joblib')
    joblib.dump(model, model_path)

    # Save feature list
    feature_path = os.path.join(output_dir, f'{model_name}_features.json')
    with open(feature_path, 'w') as f:
        json.dump(feature_cols, f, indent=2)

    # Save parameters
    params_path = os.path.join(output_dir, f'{model_name}_params.json')
    with open(params_path, 'w') as f:
        json.dump(best_params, f, indent=2, default=str)

    # Save metadata
    metadata = {
        'model_name': model_name,
        'timestamp': timestamp,
        'model_type': type(model).__name__,
        'n_features': len(feature_cols),
        'feature_names': feature_cols,
        'best_params': best_params,
        'model_path': model_path,
        'feature_path': feature_path,
        'params_path': params_path
    }

    metadata_path = os.path.join(output_dir, f'{model_name}_metadata.json')
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2, default=str)

    print(f"\n[ARTIFACTS] Model saved to: {model_path}")
    print(f"[ARTIFACTS] Features saved to: {feature_path}")
    print(f"[ARTIFACTS] Params saved to: {params_path}")
    print(f"[ARTIFACTS] Metadata saved to: {metadata_path}")

    return metadata

# Save current model artifacts
print("\n" + "="*60)
print("SAVING MODEL ARTIFACTS")
print("="*60)

# Get feature columns from training data
feature_cols = list(X_train.columns) if hasattr(X_train, 'columns') else [f'feature_{i}' for i in range(X_train_scaled.shape[1])]

# Get best params if available, otherwise use current model params
best_params = getattr(model, 'get_params', lambda: {})() if hasattr(model, 'get_params') else {}

artifacts = save_model_artifacts(
    model=model,
    feature_cols=feature_cols,
    best_params=best_params,
    model_name="xgboost_ids"
)

mem_monitor.log(force=True)


SAVING MODEL ARTIFACTS

[ARTIFACTS] Model saved to: /kaggle/working/models/xgboost_ids_20260511_201426.joblib
[ARTIFACTS] Features saved to: /kaggle/working/models/xgboost_ids_features.json
[ARTIFACTS] Params saved to: /kaggle/working/models/xgboost_ids_params.json
[ARTIFACTS] Metadata saved to: /kaggle/working/models/xgboost_ids_metadata.json
2026-05-11 20:14:26,690 [INFO] [EVENT] memory_status | {"ram_used_gb": 2.8, "ram_available_gb": 29.8, "process_rss_gb": 2.63, "peak_ram_gb": 2.63, "ram_percent": 4.9}


{'total_gb': 31.350616455078125,
 'available_gb': 29.80466079711914,
 'used_gb': 2.795757293701172,
 'process_rss_gb': 2.6266708374023438,
 'process_vms_gb': 7.996013641357422,
 'percent': 4.9}

In [19]:
# ==============================================================================
# SECTION 15 — FINAL SUMMARY & REPORT
# ==============================================================================

from datetime import datetime

def print_final_summary(model, X_train_scaled, X_val_scaled, X_test_scaled, 
                        y_train, y_val, y_test, training_time=None):
    """
    Print comprehensive final summary of the IDS pipeline.
    """
    print("\n" + "="*70)
    print("FINAL RESEARCH SUMMARY")
    print("="*70)

    # Dataset statistics
    print("\n[DATASET STATISTICS]")
    print(f"  Training samples: {len(y_train):,}")
    print(f"  Validation samples: {len(y_val):,}")
    print(f"  Test samples: {len(y_test):,}")
    print(f"  Features: {X_train_scaled.shape[1]}")

    # Class distribution
    train_counts = np.bincount(y_train.astype(int))
    val_counts = np.bincount(y_val.astype(int))
    test_counts = np.bincount(y_test.astype(int))

    print(f"  Train class distribution: {train_counts}")
    print(f"  Val class distribution: {val_counts}")
    print(f"  Test class distribution: {test_counts}")

    # Model info
    print("\n[MODEL INFORMATION]")
    print(f"  Model type: {type(model).__name__}")
    print(f"  Parameters: {model.get_params() if hasattr(model, 'get_params') else 'N/A'}")
    if training_time:
        print(f"  Training time: {training_time:.1f}s")

    # Evaluate on all splits
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

    splits = [
        ('Training', X_train_scaled, y_train),
        ('Validation', X_val_scaled, y_val),
        ('Test', X_test_scaled, y_test)
    ]

    print("\n[MODEL PERFORMANCE]")

    all_metrics = {}
    for split_name, X_split, y_split in splits:
        y_proba = model.predict_proba(X_split)[:, 1]
        y_pred = (y_proba >= 0.5).astype(int)

        acc = accuracy_score(y_split, y_pred)
        prec = precision_score(y_split, y_pred, zero_division=0)
        rec = recall_score(y_split, y_pred, zero_division=0)
        f1 = f1_score(y_split, y_pred, zero_division=0)
        roc = roc_auc_score(y_split, y_proba)

        all_metrics[split_name] = {
            'accuracy': acc,
            'precision': prec,
            'recall': rec,
            'f1': f1,
            'roc_auc': roc
        }

        print(f"\n  {split_name} Set:")
        print(f"    Accuracy:  {acc:.4f}")
        print(f"    Precision: {prec:.4f}")
        print(f"    Recall:    {rec:.4f}")
        print(f"    F1 Score:  {f1:.4f}")
        print(f"    ROC-AUC:   {roc:.4f}")

        # Confusion matrix
        from sklearn.metrics import confusion_matrix
        cm = confusion_matrix(y_split, y_pred)
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            print(f"    Confusion: TN={tn}, FP={fp}, FN={fn}, TP={tp}")

    # Feature importance (if available)
    if hasattr(model, 'feature_importances_'):
        print("\n[TOP 10 FEATURES]")
        importances = model.feature_importances_
        feature_names = list(X_train.columns) if hasattr(X_train, 'columns') else [f'f{i}' for i in range(len(importances))]

        top_idx = np.argsort(importances)[::-1][:10]
        for i, idx in enumerate(top_idx, 1):
            print(f"  {i:2d}. {feature_names[idx]:20s}: {importances[idx]:.4f}")

    # Save summary to file
    summary = {
        'timestamp': datetime.now().isoformat(),
        'dataset': {
            'train_size': len(y_train),
            'val_size': len(y_val),
            'test_size': len(y_test),
            'n_features': X_train_scaled.shape[1]
        },
        'model': {
            'type': type(model).__name__,
            'params': model.get_params() if hasattr(model, 'get_params') else {}
        },
        'metrics': all_metrics
    }

    summary_path = os.path.join(WORKING_PATH, 'final_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2, default=str)

    print(f"\n[INFO] Summary saved to: {summary_path}")
    print("="*70)

    return summary

# Execute final summary
print("\n" + "="*70)
print("GENERATING FINAL SUMMARY")
print("="*70)

# FIX: Pass all required variables directly
summary = print_final_summary(
    model=model,
    X_train_scaled=X_train_scaled,
    X_val_scaled=X_val_scaled,
    X_test_scaled=X_test_scaled,
    y_train=y_train,
    y_val=y_val,
    y_test=y_test,
    training_time=None  # Or pass actual training time if available
)

mem_monitor.log(force=True)


GENERATING FINAL SUMMARY

FINAL RESEARCH SUMMARY

[DATASET STATISTICS]
  Training samples: 9,442
  Validation samples: 4,720
  Test samples: 4,720
  Features: 43
  Train class distribution: [4721 4721]
  Val class distribution: [2360 2360]
  Test class distribution: [2360 2360]

[MODEL INFORMATION]
  Model type: XGBClassifier
  Parameters: {'objective': 'binary:logistic', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': 0.6566993417872399, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': False, 'eval_metric': 'logloss', 'feature_types': None, 'feature_weights': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.04113261676831118, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 6, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraint

{'total_gb': 31.350616455078125,
 'available_gb': 29.804767608642578,
 'used_gb': 2.7954368591308594,
 'process_rss_gb': 2.6267967224121094,
 'process_vms_gb': 7.996013641357422,
 'percent': 4.9}

In [20]:
# ==============================================================================
# BONUS: ALL VISUALIZATIONS FOR IDS PIPELINE
# ==============================================================================

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (roc_curve, precision_recall_curve, 
                             confusion_matrix, ConfusionMatrixDisplay)
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Create output directory
os.makedirs(WORKING_PATH, exist_ok=True)

print("\n" + "="*60)
print("GENERATING ALL VISUALIZATIONS")
print("="*60)

# -----------------------------------------------------------------------------
# 1. ROC & PR CURVES (Combined)
# -----------------------------------------------------------------------------
print("\n[1/7] Generating ROC & PR Curves...")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

splits = [
    ('Training', y_train, model.predict_proba(X_train_scaled)[:, 1]),
    ('Validation', y_val, model.predict_proba(X_val_scaled)[:, 1]),
    ('Test', y_test, model.predict_proba(X_test_scaled)[:, 1])
]

colors = ['#2E86AB', '#A23B72', '#F18F01']

for i, (name, y_true, y_proba) in enumerate(splits):
    # ROC
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    auc = roc_auc_score(y_true, y_proba)
    axes[0].plot(fpr, tpr, color=colors[i], linewidth=2.5, 
                label=f'{name} (AUC={auc:.4f})')

    # PR
    precision, recall, _ = precision_recall_curve(y_true, y_proba)
    ap = average_precision_score(y_true, y_proba)
    axes[1].plot(recall, precision, color=colors[i], linewidth=2.5,
                label=f'{name} (AP={ap:.4f})')

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3, linewidth=1)
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('ROC Curves', fontsize=14, fontweight='bold')
axes[0].legend(loc='lower right', fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title('Precision-Recall Curves', fontsize=14, fontweight='bold')
axes[1].legend(loc='lower left', fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(WORKING_PATH, 'roc_pr_curves.png'), dpi=300, bbox_inches='tight')
plt.close()
print("  ✓ roc_pr_curves.png")

# -----------------------------------------------------------------------------
# 2. FEATURE IMPORTANCE (Horizontal Bar)
# -----------------------------------------------------------------------------
print("\n[2/7] Generating Feature Importance...")

if hasattr(model, 'feature_importances_'):
    importances = model.feature_importances_
    feature_names = list(X_train.columns) if hasattr(X_train, 'columns') else [f'f{i}' for i in range(len(importances))]

    # Top 15 features
    top_n = 15
    top_idx = np.argsort(importances)[::-1][:top_n]
    top_features = [feature_names[i] for i in top_idx]
    top_values = importances[top_idx]

    fig, ax = plt.subplots(figsize=(10, 8))
    colors_bar = plt.cm.viridis(np.linspace(0.2, 0.8, top_n))
    bars = ax.barh(range(top_n), top_values[::-1], color=colors_bar[::-1])

    ax.set_yticks(range(top_n))
    ax.set_yticklabels(top_features[::-1], fontsize=11)
    ax.set_xlabel('Importance Score', fontsize=12)
    ax.set_title(f'Top {top_n} Feature Importances', fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

    # Add value labels
    for i, (bar, val) in enumerate(zip(bars, top_values[::-1])):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2, 
                f'{val:.3f}', va='center', fontsize=9)

    plt.tight_layout()
    plt.savefig(os.path.join(WORKING_PATH, 'feature_importance.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print("  ✓ feature_importance.png")
else:
    print("  ✗ Model has no feature_importances_")

# -----------------------------------------------------------------------------
# 3. CONFUSION MATRICES (All Splits)
# -----------------------------------------------------------------------------
print("\n[3/7] Generating Confusion Matrices...")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

split_data = [
    ('Training', y_train, X_train_scaled),
    ('Validation', y_val, X_val_scaled),
    ('Test', y_test, X_test_scaled)
]

for i, (name, y_true, X_split) in enumerate(split_data):
    y_pred = model.predict(X_split)
    cm = confusion_matrix(y_true, y_pred)

    disp = ConfusionMatrixDisplay(cm, display_labels=['Benign', 'Attack'])
    disp.plot(ax=axes[i], cmap='Blues', values_format='d', colorbar=False)
    axes[i].set_title(f'{name} Set', fontsize=13, fontweight='bold')

    # Add accuracy text
    acc = accuracy_score(y_true, y_pred)
    axes[i].text(0.5, -0.15, f'Accuracy: {acc:.4f}', 
                transform=axes[i].transAxes, ha='center', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(WORKING_PATH, 'confusion_matrices.png'), dpi=300, bbox_inches='tight')
plt.close()
print("  ✓ confusion_matrices.png")

# -----------------------------------------------------------------------------
# 4. PREDICTION DISTRIBUTION (Histogram)
# -----------------------------------------------------------------------------
print("\n[4/7] Generating Prediction Distribution...")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, (name, y_true, X_split) in enumerate(split_data):
    y_proba = model.predict_proba(X_split)[:, 1]

    benign_proba = y_proba[y_true == 0]
    attack_proba = y_proba[y_true == 1]

    axes[i].hist(benign_proba, bins=30, alpha=0.6, label='Benign', color='#2E86AB', density=True)
    axes[i].hist(attack_proba, bins=30, alpha=0.6, label='Attack', color='#F18F01', density=True)
    axes[i].axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Threshold=0.5')
    axes[i].set_xlabel('Predicted Probability (Attack)', fontsize=11)
    axes[i].set_ylabel('Density', fontsize=11)
    axes[i].set_title(f'{name} Set', fontsize=13, fontweight='bold')
    axes[i].legend(fontsize=9)
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(WORKING_PATH, 'prediction_distribution.png'), dpi=300, bbox_inches='tight')
plt.close()
print("  ✓ prediction_distribution.png")

# -----------------------------------------------------------------------------
# 5. THRESHOLD vs F1 CURVE
# -----------------------------------------------------------------------------
print("\n[5/7] Generating Threshold-F1 Curve...")

thresholds = np.arange(0.01, 1.0, 0.01)
f1_scores = []
precisions = []
recalls = []

y_val_proba = model.predict_proba(X_val_scaled)[:, 1]

for thresh in thresholds:
    y_pred = (y_val_proba >= thresh).astype(int)
    f1_scores.append(f1_score(y_val, y_pred, zero_division=0))
    precisions.append(precision_score(y_val, y_pred, zero_division=0))
    recalls.append(recall_score(y_val, y_pred, zero_division=0))

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(thresholds, f1_scores, linewidth=2.5, color='#2E86AB', label='F1 Score')
ax.plot(thresholds, precisions, linewidth=2.5, color='#A23B72', label='Precision', alpha=0.7)
ax.plot(thresholds, recalls, linewidth=2.5, color='#F18F01', label='Recall', alpha=0.7)

# Mark optimal threshold
best_idx = np.argmax(f1_scores)
ax.axvline(x=thresholds[best_idx], color='red', linestyle='--', linewidth=2, 
          label=f'Optimal: {thresholds[best_idx]:.2f}')
ax.scatter([thresholds[best_idx]], [f1_scores[best_idx]], color='red', s=100, zorder=5)

ax.set_xlabel('Threshold', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Threshold vs. Performance Metrics (Validation Set)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(os.path.join(WORKING_PATH, 'threshold_f1_curve.png'), dpi=300, bbox_inches='tight')
plt.close()
print("  ✓ threshold_f1_curve.png")

# -----------------------------------------------------------------------------
# 6. METRICS COMPARISON BAR CHART
# -----------------------------------------------------------------------------
print("\n[6/7] Generating Metrics Comparison...")

metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
train_metrics = [
    accuracy_score(y_train, model.predict(X_train_scaled)),
    precision_score(y_train, model.predict(X_train_scaled), zero_division=0),
    recall_score(y_train, model.predict(X_train_scaled), zero_division=0),
    f1_score(y_train, model.predict(X_train_scaled), zero_division=0),
    roc_auc_score(y_train, model.predict_proba(X_train_scaled)[:, 1])
]
val_metrics = [
    accuracy_score(y_val, model.predict(X_val_scaled)),
    precision_score(y_val, model.predict(X_val_scaled), zero_division=0),
    recall_score(y_val, model.predict(X_val_scaled), zero_division=0),
    f1_score(y_val, model.predict(X_val_scaled), zero_division=0),
    roc_auc_score(y_val, model.predict_proba(X_val_scaled)[:, 1])
]
test_metrics = [
    accuracy_score(y_test, model.predict(X_test_scaled)),
    precision_score(y_test, model.predict(X_test_scaled), zero_division=0),
    recall_score(y_test, model.predict(X_test_scaled), zero_division=0),
    f1_score(y_test, model.predict(X_test_scaled), zero_division=0),
    roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1])
]

x = np.arange(len(metrics_names))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
rects1 = ax.bar(x - width, train_metrics, width, label='Train', color='#2E86AB')
rects2 = ax.bar(x, val_metrics, width, label='Val', color='#A23B72')
rects3 = ax.bar(x + width, test_metrics, width, label='Test', color='#F18F01')

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Performance Metrics Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0.95, 1.005)
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for rects in [rects1, rects2, rects3]:
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.4f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points",
                    ha='center', va='bottom', fontsize=8, rotation=90)

plt.tight_layout()
plt.savefig(os.path.join(WORKING_PATH, 'metrics_comparison.png'), dpi=300, bbox_inches='tight')
plt.close()
print("  ✓ metrics_comparison.png")

# -----------------------------------------------------------------------------
# 7. CLASS DISTRIBUTION (Pie Charts)
# -----------------------------------------------------------------------------
print("\n[7/7] Generating Class Distribution...")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

datasets = [
    ('Training', y_train),
    ('Validation', y_val),
    ('Test', y_test)
]

colors_pie = ['#2E86AB', '#F18F01']

for i, (name, y_data) in enumerate(datasets):
    counts = np.bincount(y_data.astype(int))
    axes[i].pie(counts, labels=['Benign', 'Attack'], autopct='%1.1f%%',
               colors=colors_pie, startangle=90, textprops={'fontsize': 11})
    axes[i].set_title(f'{name} Set', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(WORKING_PATH, 'class_distribution.png'), dpi=300, bbox_inches='tight')
plt.close()
print("  ✓ class_distribution.png")

# -----------------------------------------------------------------------------
# SUMMARY
# -----------------------------------------------------------------------------
print("\n" + "="*60)
print("ALL VISUALIZATIONS COMPLETE")
print("="*60)

viz_files = [
    'roc_pr_curves.png',
    'feature_importance.png',
    'confusion_matrices.png',
    'prediction_distribution.png',
    'threshold_f1_curve.png',
    'metrics_comparison.png',
    'class_distribution.png'
]

print("\nGenerated files:")
for f in viz_files:
    path = os.path.join(WORKING_PATH, f)
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f"  ✓ {f} ({size:,} bytes)")
    else:
        print(f"  ✗ {f} (missing)")


GENERATING ALL VISUALIZATIONS

[1/7] Generating ROC & PR Curves...
  ✓ roc_pr_curves.png

[2/7] Generating Feature Importance...
  ✓ feature_importance.png

[3/7] Generating Confusion Matrices...
  ✓ confusion_matrices.png

[4/7] Generating Prediction Distribution...
  ✓ prediction_distribution.png

[5/7] Generating Threshold-F1 Curve...
  ✓ threshold_f1_curve.png

[6/7] Generating Metrics Comparison...
  ✓ metrics_comparison.png

[7/7] Generating Class Distribution...
  ✓ class_distribution.png

ALL VISUALIZATIONS COMPLETE

Generated files:
  ✓ roc_pr_curves.png (196,757 bytes)
  ✓ feature_importance.png (198,871 bytes)
  ✓ confusion_matrices.png (113,109 bytes)
  ✓ prediction_distribution.png (130,156 bytes)
  ✓ threshold_f1_curve.png (129,494 bytes)
  ✓ metrics_comparison.png (134,172 bytes)
  ✓ class_distribution.png (103,690 bytes)


In [21]:
# ==============================================================================
# SAVE MODEL WEIGHTS & ARTIFACTS
# ==============================================================================

import joblib
import json
import pickle
import os

print("\n" + "="*60)
print("SAVING MODEL WEIGHTS & ARTIFACTS")
print("="*60)

# Ensure directories exist
os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(WORKING_PATH, exist_ok=True)

# -----------------------------------------------------------------------------
# 1. SAVE FULL MODEL (joblib - recommended for sklearn/xgboost)
# -----------------------------------------------------------------------------
model_path = os.path.join(MODEL_PATH, 'xgboost_ids_model.joblib')
joblib.dump(model, model_path)
print(f"[1/7] Full model saved: {model_path}")

# Verify
loaded_model = joblib.load(model_path)
test_pred = loaded_model.predict(X_test_scaled[:10])
verify_acc = accuracy_score(y_test[:10], test_pred)
print(f"       Verification: {verify_acc:.4f} accuracy on 10 samples")

# -----------------------------------------------------------------------------
# 2. SAVE MODEL PARAMETERS ONLY (JSON - human readable)
# -----------------------------------------------------------------------------
params_path = os.path.join(MODEL_PATH, 'model_params.json')
model_params = model.get_params()

# Convert numpy types to native Python for JSON serialization
def convert_to_native(obj):
    if hasattr(obj, 'item'):  # numpy scalar
        return obj.item()
    elif isinstance(obj, dict):
        return {k: convert_to_native(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_native(i) for i in obj]
    return obj

with open(params_path, 'w') as f:
    json.dump(convert_to_native(model_params), f, indent=2)
print(f"[2/7] Model params saved: {params_path}")

# -----------------------------------------------------------------------------
# 3. SAVE FEATURE IMPORTANCES (CSV)
# -----------------------------------------------------------------------------
if hasattr(model, 'feature_importances_'):
    import pandas as pd

    feature_names = list(X_train.columns) if hasattr(X_train, 'columns') else [f'f{i}' for i in range(len(model.feature_importances_))]

    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)

    importance_path = os.path.join(MODEL_PATH, 'feature_importances.csv')
    importance_df.to_csv(importance_path, index=False)
    print(f"[3/7] Feature importances saved: {importance_path}")

# -----------------------------------------------------------------------------
# 4. SAVE SCALER (for inference preprocessing)
# -----------------------------------------------------------------------------
scaler_path = os.path.join(MODEL_PATH, 'scaler.joblib')
joblib.dump(scaler, scaler_path)
print(f"[4/7] Scaler saved: {scaler_path}")

# -----------------------------------------------------------------------------
# 5. SAVE FEATURE LIST (for inference validation)
# -----------------------------------------------------------------------------
feature_list_path = os.path.join(MODEL_PATH, 'feature_list.json')
feature_list = list(X_train.columns) if hasattr(X_train, 'columns') else [f'f{i}' for i in range(X_train_scaled.shape[1])]

with open(feature_list_path, 'w') as f:
    json.dump(feature_list, f, indent=2)
print(f"[5/7] Feature list saved: {feature_list_path}")

# -----------------------------------------------------------------------------
# 6. SAVE LABEL ENCODER (if categorical features were encoded)
# -----------------------------------------------------------------------------
# If you used LabelEncoder in Section 8, save them too
# Example:
# le_path = os.path.join(MODEL_PATH, 'label_encoders.joblib')
# joblib.dump(label_encoders_dict, le_path)
# print(f"[6/7] Label encoders saved: {le_path}")

# -----------------------------------------------------------------------------
# 7. SAVE COMPLETE PIPELINE (model + scaler + metadata)
# -----------------------------------------------------------------------------
pipeline = {
    'model': model,
    'scaler': scaler,
    'feature_list': feature_list,
    'model_params': model_params,
    'training_info': {
        'train_samples': len(y_train),
        'val_samples': len(y_val),
        'test_samples': len(y_test),
        'n_features': len(feature_list),
        'accuracy': {
            'train': accuracy_score(y_train, model.predict(X_train_scaled)),
            'val': accuracy_score(y_val, model.predict(X_val_scaled)),
            'test': accuracy_score(y_test, model.predict(X_test_scaled))
        }
    }
}

pipeline_path = os.path.join(MODEL_PATH, 'complete_pipeline.joblib')
joblib.dump(pipeline, pipeline_path)
print(f"[6/7] Complete pipeline saved: {pipeline_path}")

# -----------------------------------------------------------------------------
# 8. SAVE XGBOOST NATIVE FORMAT (for deployment / C++ inference)
# -----------------------------------------------------------------------------
if hasattr(model, 'save_model'):
    xgb_native_path = os.path.join(MODEL_PATH, 'xgboost_model.json')
    model.save_model(xgb_native_path)
    print(f"[7/7] XGBoost native format: {xgb_native_path}")

    # This can be loaded with:
    # loaded_xgb = xgb.XGBClassifier()
    # loaded_xgb.load_model(xgb_native_path)

# -----------------------------------------------------------------------------
# SUMMARY
# -----------------------------------------------------------------------------
print("\n" + "="*60)
print("ALL ARTIFACTS SAVED")
print("="*60)
print(f"\nModel directory: {MODEL_PATH}")
print(f"Files saved:")
for f in os.listdir(MODEL_PATH):
    fpath = os.path.join(MODEL_PATH, f)
    size = os.path.getsize(fpath)
    print(f"  {f} ({size:,} bytes)")

print("\n[USAGE] Load model for inference:")
print("  model = joblib.load('" + model_path + "')")
print("  scaler = joblib.load('" + scaler_path + "')")
print("  # Preprocess new data with scaler, then predict")


SAVING MODEL WEIGHTS & ARTIFACTS
[1/7] Full model saved: /kaggle/working/models/xgboost_ids_model.joblib
       Verification: 1.0000 accuracy on 10 samples
[2/7] Model params saved: /kaggle/working/models/model_params.json
[3/7] Feature importances saved: /kaggle/working/models/feature_importances.csv
[4/7] Scaler saved: /kaggle/working/models/scaler.joblib
[5/7] Feature list saved: /kaggle/working/models/feature_list.json
[6/7] Complete pipeline saved: /kaggle/working/models/complete_pipeline.joblib
[7/7] XGBoost native format: /kaggle/working/models/xgboost_model.json

ALL ARTIFACTS SAVED

Model directory: /kaggle/working/models/
Files saved:
  xgboost_ids_metadata.json (2,183 bytes)
  feature_importances.csv (827 bytes)
  xgboost_ids_params.json (1,074 bytes)
  feature_list.json (563 bytes)
  complete_pipeline.joblib (167,891 bytes)
  xgboost_ids_features.json (563 bytes)
  deployment_artifacts.json (2,537 bytes)
  xgboost_model.json (204,600 bytes)
  scaler.joblib (1,871 bytes)
  

In [22]:
import gradio as gr
import joblib
import numpy as np
import pandas as pd
import json
import os

# Ngarko modelin dhe scaler
MODEL_PATH = "/kaggle/working/models/xgboost_ids_model.joblib"
SCALER_PATH = "/kaggle/working/models/scaler.joblib"
FEATURES_PATH = "/kaggle/working/models/feature_list.json"

model = joblib.load(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)

with open(FEATURES_PATH) as f:
    feature_list = json.load(f)

def predict_intrusion(*features):
    """
    Merr veçoritë si input, kthen prediksionin.
    """
    # Krijo DataFrame
    input_df = pd.DataFrame([features], columns=feature_list)
    
    # Scale
    input_scaled = scaler.transform(input_df)
    
    # Predikto
    proba = model.predict_proba(input_scaled)[0, 1]
    pred = 1 if proba >= 0.5 else 0
    
    # Rezultati
    if pred == 1:
        return f"🚨 SULM (Attack) - Probabiliteti: {proba:.4f}"
    else:
        return f"✅ NORMAL (Benign) - Probabiliteti: {proba:.4f}"

# Krijo inputet dinamikisht bazuar në feature_list
inputs = []
for feat in feature_list:
    inputs.append(gr.Number(label=feat, value=0.0))

# Krijo interface
demo = gr.Interface(
    fn=predict_intrusion,
    inputs=inputs,
    outputs=gr.Textbox(label="Rezultati", lines=2),
    title="🔒 CYBERJACK INTRUSION DETECTION SYSTEM",
    description="Set network flow property values ​​to detect attacks.",
    examples=[[0.0] * len(feature_list)]  # Shembull me zero
)

demo.launch()

2026-05-11 20:14:42,255 [INFO] HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
* Running on local URL:  http://127.0.0.1:7860
2026-05-11 20:14:42,988 [INFO] HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

2026-05-11 20:14:43,054 [INFO] HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"
2026-05-11 20:14:43,060 [INFO] HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-05-11 20:14:43,659 [INFO] HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"
2026-05-11 20:14:43,742 [INFO] HTTP Request: GET https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_linux_amd64 "HTTP/1.1 200 OK"
* Running on public URL: https://a9f4e48f

In [23]:
# ==============================================================================
# MODEL VALIDATION & OVERFITTING/UNDERFITTING CHECK (FINAL)
# ==============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score, StratifiedKFold, learning_curve
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, log_loss, brier_score_loss)
from sklearn.calibration import calibration_curve
from sklearn.dummy import DummyClassifier
import json

print("="*70)
print("MODEL VALIDATION & OVERFITTING/UNDERFITTING ANALYSIS")
print("="*70)

# ==============================================================================
# 1. BASIC METRICS COMPARISON
# ==============================================================================
print("\n[1/10] BASIC METRICS COMPARISON")
print("-"*70)

def evaluate_model(model, X, y, dataset_name):
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]
    return {
        'dataset': dataset_name,
        'accuracy': accuracy_score(y, y_pred),
        'precision': precision_score(y, y_pred, zero_division=0),
        'recall': recall_score(y, y_pred, zero_division=0),
        'f1': f1_score(y, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y, y_proba),
        'log_loss': log_loss(y, y_proba),
        'n_samples': len(y)
    }

results = []
for name, X, y in [('Train', X_train_scaled, y_train),
                    ('Val', X_val_scaled, y_val),
                    ('Test', X_test_scaled, y_test)]:
    results.append(evaluate_model(model, X, y, name))

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# Overfitting check
train_acc = results_df[results_df['dataset'] == 'Train']['accuracy'].values[0]
test_acc = results_df[results_df['dataset'] == 'Test']['accuracy'].values[0]
gap = train_acc - test_acc

print(f"\n[OVERFITTING CHECK]")
print(f"Train-Test Accuracy Gap: {gap:.4f} ({gap*100:.2f}%)")
if gap < 0.02:
    print("✅ PASS: Gap < 2% — No overfitting")
elif gap < 0.05:
    print("⚠️  WARNING: Gap 2-5% — Mild overfitting")
else:
    print("❌ FAIL: Gap > 5% — Significant overfitting")

# ==============================================================================
# 2. CROSS-VALIDATION (5-Fold)
# ==============================================================================
print("\n[2/10] 5-FOLD STRATIFIED CROSS-VALIDATION")
print("-"*70)

X_cv = np.vstack([X_train_scaled, X_val_scaled])
y_cv = np.concatenate([y_train, y_val])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_cv, y_cv, cv=cv, scoring='accuracy')

print(f"CV Scores: {cv_scores}")
print(f"Mean CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"Test Accuracy: {test_acc:.4f}")

cv_test_diff = abs(cv_scores.mean() - test_acc)
print(f"CV-Test Difference: {cv_test_diff:.4f}")

if cv_test_diff < 0.02:
    print("✅ PASS: CV and Test agree")
else:
    print("⚠️  WARNING: CV and Test differ")

# ==============================================================================
# 3. LEARNING CURVES
# ==============================================================================
print("\n[3/10] LEARNING CURVES")
print("-"*70)

train_sizes, train_scores, val_scores = learning_curve(
    model, X_cv, y_cv, cv=cv, scoring='accuracy',
    train_sizes=np.linspace(0.1, 1.0, 10), random_state=42
)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_scores.mean(axis=1), 'o-', color='blue', label='Training')
plt.plot(train_sizes, val_scores.mean(axis=1), 'o-', color='red', label='Validation')
plt.fill_between(train_sizes, train_scores.mean(axis=1) - train_scores.std(axis=1),
                 train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.1, color='blue')
plt.fill_between(train_sizes, val_scores.mean(axis=1) - val_scores.std(axis=1),
                 val_scores.mean(axis=1) + val_scores.std(axis=1), alpha=0.1, color='red')
plt.xlabel('Training Set Size')
plt.ylabel('Accuracy')
plt.title('Learning Curves')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('/kaggle/working/learning_curves.png', dpi=300, bbox_inches='tight')
plt.close()
print("✅ Saved: learning_curves.png")

# ==============================================================================
# 4. NOISE ROBUSTNESS TEST
# ==============================================================================
print("\n[4/10] NOISE ROBUSTNESS TEST")
print("-"*70)

noise_levels = [0.0, 0.01, 0.05, 0.1, 0.2]
noise_results = []

for noise_std in noise_levels:
    np.random.seed(42)
    noise = np.random.normal(0, noise_std, X_test_scaled.shape)
    X_noisy = X_test_scaled + noise
    y_noisy_pred = model.predict(X_noisy)
    acc = accuracy_score(y_test, y_noisy_pred)
    noise_results.append({'noise_std': noise_std, 'accuracy': acc})
    print(f"  Noise std={noise_std:.2f}: Accuracy={acc:.4f}")

noise_df = pd.DataFrame(noise_results)
max_drop = noise_df['accuracy'].iloc[0] - noise_df['accuracy'].min()
print(f"\nMax accuracy drop with noise: {max_drop:.4f}")

if max_drop < 0.10:
    print("✅ PASS: Robust to noise")
elif max_drop < 0.15:
    print("⚠️  WARNING: Moderate sensitivity")
else:
    print("❌ FAIL: Very sensitive to noise")

plt.figure(figsize=(8, 5))
plt.plot(noise_df['noise_std'], noise_df['accuracy'], 'o-', linewidth=2)
plt.xlabel('Noise Standard Deviation')
plt.ylabel('Accuracy')
plt.title('Noise Robustness Test')
plt.grid(True, alpha=0.3)
plt.savefig('/kaggle/working/noise_robustness.png', dpi=300, bbox_inches='tight')
plt.close()
print("✅ Saved: noise_robustness.png")

# ==============================================================================
# 5. FEATURE IMPORTANCE STABILITY
# ==============================================================================
print("\n[5/10] FEATURE IMPORTANCE STABILITY")
print("-"*70)

if hasattr(model, 'feature_importances_'):
    importances = model.feature_importances_
    feature_names = list(X_train.columns) if hasattr(X_train, 'columns') else [f'f{i}' for i in range(len(importances))]
    
    max_imp = importances.max()
    max_idx = importances.argmax()
    
    print(f"Top feature: {feature_names[max_idx]} ({max_imp:.3f})")
    print(f"Top 3 sum: {np.sort(importances)[-3:].sum():.3f}")
    print(f"Top 10 sum: {np.sort(importances)[-10:].sum():.3f}")
    
    if max_imp > 0.5:
        print("⚠️  WARNING: One feature dominates >50%")
    elif max_imp > 0.3:
        print("ℹ️  INFO: One feature >30%")
    else:
        print("✅ PASS: Importances well distributed")

# ==============================================================================
# 6. PERMUTATION TEST (Random Baseline)
# ==============================================================================
print("\n[6/10] PERMUTATION TEST")
print("-"*70)

dummy = DummyClassifier(strategy='stratified', random_state=42)
dummy.fit(X_train_scaled, np.random.permutation(y_train))
dummy_acc = dummy.score(X_test_scaled, y_test)

print(f"Random classifier: {dummy_acc:.4f}")
print(f"Your model: {test_acc:.4f}")
print(f"Improvement: {(test_acc - dummy_acc) / dummy_acc * 100:.1f}%")

if test_acc > dummy_acc + 0.1:
    print("✅ PASS: Significantly better than random")
else:
    print("❌ FAIL: Barely better than random")

# ==============================================================================
# 7. CALIBRATION CHECK
# ==============================================================================
print("\n[7/10] CALIBRATION CHECK")
print("-"*70)

y_test_proba = model.predict_proba(X_test_scaled)[:, 1]
fraction_of_positives, mean_predicted_value = calibration_curve(y_test, y_test_proba, n_bins=10)

plt.figure(figsize=(8, 6))
plt.plot(mean_predicted_value, fraction_of_positives, 's-', label='Model')
plt.plot([0, 1], [0, 1], 'k--', label='Perfect')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.title('Calibration Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('/kaggle/working/calibration_curve.png', dpi=300, bbox_inches='tight')
plt.close()
print("✅ Saved: calibration_curve.png")

brier = brier_score_loss(y_test, y_test_proba)
print(f"Brier Score: {brier:.4f} (0=perfect, 0.25=random)")

if brier < 0.05:
    print("✅ PASS: Well calibrated")
elif brier < 0.1:
    print("ℹ️  INFO: Reasonably calibrated")
else:
    print("⚠️  WARNING: Poorly calibrated")

# ==============================================================================
# 8. FINAL VERDICT
# ==============================================================================
print("\n[8/10] FINAL VERDICT")
print("="*70)

summary = {
    'overfitting_detected': gap > 0.02,
    'underfitting_detected': test_acc < 0.9,
    'train_test_gap': float(gap),
    'cv_mean_accuracy': float(cv_scores.mean()),
    'cv_std': float(cv_scores.std()),
    'test_accuracy': float(test_acc),
    'noise_robustness_max_drop': float(max_drop),
    'calibration_brier_score': float(brier),
    'model_is_valid': (gap < 0.02) and (cv_test_diff < 0.02) and (max_drop < 0.05) and (test_acc > 0.95)
}

for key, value in summary.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

print("\n" + "="*70)
if summary['model_is_valid']:
    print("✅✅✅ MODEL IS VALID — NO OVERFITTING, NO UNDERFITTING ✅✅✅")
    print("The model generalizes well and is ready for deployment.")
else:
    print("⚠️⚠️⚠️ MODEL NEEDS ATTENTION ⚠️⚠️⚠️")
print("="*70)

# Save report
with open('/kaggle/working/model_validation_report.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print("\n✅ Saved: model_validation_report.json")

MODEL VALIDATION & OVERFITTING/UNDERFITTING ANALYSIS

[1/10] BASIC METRICS COMPARISON
----------------------------------------------------------------------
2026-05-11 20:14:45,256 [INFO] HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"
dataset  accuracy  precision   recall       f1  roc_auc  log_loss  n_samples
  Train  0.999788   1.000000 0.999576 0.999788 0.999999  0.012839       9442
    Val  0.999153   0.999576 0.998729 0.999152 0.999933  0.015779       4720
   Test  0.998941   0.999576 0.998305 0.998940 0.999991  0.014099       4720

[OVERFITTING CHECK]
Train-Test Accuracy Gap: 0.0008 (0.08%)
✅ PASS: Gap < 2% — No overfitting

[2/10] 5-FOLD STRATIFIED CROSS-VALIDATION
----------------------------------------------------------------------
CV Scores: [0.99858807 0.9978821  0.99858757 0.99929379 0.99929379]
Mean CV Accuracy: 0.9987 (+/- 0.0005)
Test Accuracy: 0.9989
CV-Test Difference: 0.0002
✅ PASS: CV and T